In [ ]:
# Step 1: Generate 2D Head Dataset and Velocity Dataset
import numpy as np
import os
import pandas as pd
import h5py
from numba import jit, prange

# Parameter settings
params = {
    'p': 3, 'q': 3, 'HRx': 6, 'HLx': 3, 'LX': 10000, 'D': -2000, 'a': 2/3, 'b': 1/3, 'TauD': 0.4
}

# Extract parameters
p, q, HRx, HLx, LX, D, a, b, TauD = params['p'], params['q'], params['HRx'], params['HLx'], params['LX'], params['D'], params['a'], params['b'], params['TauD']

# Define the mesh grid 
mesh_x, mesh_z = 201, 201
x_range = np.linspace(0, LX, mesh_x)
z_range = np.linspace(D, 0, mesh_z)

# Precompute cosine values
b1 = np.cos(np.pi * x_range / LX)
b2 = np.cos(p * np.pi * x_range / LX)

@jit(nopython=True, parallel=True)
def compute_J(mesh_x, mesh_z, x_range, z_range, HRx, HLx, LX, D, a, b, TauD, b1, b2, c1, c2, e1, e2):
    """Calculate the J matrix and related variables in 2D (x, z) space."""
    J = np.zeros((mesh_x, mesh_z))
    for i in prange(mesh_x):
        for k in prange(mesh_z):
            # Calculate potential components hA, hB
            hA = HRx - HRx * b1[i] * np.cosh(np.pi * (z_range[k] - D) / LX) / c1
            hB = HLx - HLx * b2[i] * np.cosh(p * np.pi * (z_range[k] - D) / LX) / c2
            g1, g2 = 0.0, 0.0
            for n in range(1, 1000):  
                k1, an = (-1) ** (n - 1), (2 * n - 1) * np.pi / 2
                b0 = np.cos(an * (z_range[k] - D) / D)
                f1 = D * (k1 * an * b0 * (1 / an**2 + (an**2 * e1 + 2 * np.pi * TauD * e2) / (an**4 + 4 * np.pi**2 * TauD**2)))
                g1 += f1
                k2, d1 = (-1) ** n, an**2 + (p * np.pi * D / LX)**2
                f2 = D * (k2 * an * b0 * (1 / d1 + (d1 * e1 + 2 * np.pi * TauD * e2) / (d1**2 + 4 * np.pi**2 * TauD**2)))
                g2 += f2               
            hC = HLx * g1 / D+HLx * b2[i] * g2 / D
            J[i, k] = hA + a * hB + b * hC
    return J

def calculate_fields(tp):
    """Calculate the potential field (J), velocity components (vx, vz), and velocity magnitude (UU) for a given time point (tp)."""
    e1 = np.cos(2 * np.pi * tp)
    e2 = np.sin(2 * np.pi * tp)

    # Precompute cosh values for efficiency
    c1 = np.cosh(np.pi * D / LX)
    c2 = np.cosh(p * np.pi * D / LX)

    J = compute_J(mesh_x, mesh_z, x_range, z_range, HRx, HLx, LX, D, a, b, TauD, b1, b2, c1, c2, e1, e2)

    # Compute the gradient of J (only x and z directions)
    dx, dz = x_range[1] - x_range[0], z_range[1] - z_range[0]
    grad_x, grad_z = np.gradient(J, dx, dz)

    # Calculate the velocity components as the negative gradient of J
    vx, vz = -grad_x, -grad_z

    # Calculate the magnitude of the velocity field
    UU = np.sqrt(vx**2 + vz**2)

    # Hydraulic head (H) is equivalent to J
    H = J

    return H, vx, vz, UU

def save_data_hdf5(filename, J, vx, vz, UU, H, x_range, z_range):
    """Save the computed data into an HDF5 file."""
    with h5py.File(filename, 'w') as f:
        f.create_dataset('J', data=J)
        f.create_dataset('vx', data=vx)
        f.create_dataset('vz', data=vz)
        f.create_dataset('UU', data=UU)
        f.create_dataset('H', data=H)
        f.create_dataset('x_range', data=x_range)
        f.create_dataset('z_range', data=z_range)

def save_data_tecplot(filename, vx, vz, UU, J, x_range, z_range):
    """Save the computed data into a Tecplot-compatible .dat file."""
    # Apply mask for z_range < -20
    mask = z_range < -20

    x, z = np.meshgrid(x_range, z_range, indexing='ij')
    
    # Flatten and filter data
    x = x[:, mask].flatten()
    z = z[:, mask].flatten()
    vx = vx[:, mask].flatten()
    vz = vz[:, mask].flatten()
    UU = UU[:, mask].flatten()
    J = J[:, mask].flatten()

    # Stack data column-wise
    data = np.column_stack((x, z, vx, vz, UU, J))

    # Sort data by x, then z
    data = data[np.lexsort((x, z))]

    with open(filename, 'w') as f:
        f.write('TITLE = "Velocity, Speed and Head Data"\n')
        f.write('VARIABLES = "X", "Z", "VX", "VZ", "UU", "H"\n')
        f.write(f'ZONE T="Flow Field", I={len(x_range)}, K={len(z_range[mask])}, F=POINT\n')
        np.savetxt(f, data, fmt='%.6f', delimiter=' ')

# Setup output directory
current_working_dir = os.getcwd()
result_folder = os.path.join(current_working_dir, 'results')
os.makedirs(result_folder, exist_ok=True)

# Loop over time points (tp) and compute, save results
for tp in np.arange(0, 1, 0.05):
    H, vx, vz, UU = calculate_fields(tp)

    # Construct file names for the current time point
    hdf5_filename = os.path.join(result_folder, f'head_and_velocity_tp_{tp:.2f}.h5')
    tecplot_filename = os.path.join(result_folder, f'head_and_velocity_tp_{tp:.2f}.dat')
    
    # Save data to HDF5 format
    save_data_hdf5(hdf5_filename, H, vx, vz, UU, H, x_range, z_range)
    
    # Save data to Tecplot .dat format
    save_data_tecplot(tecplot_filename, vx, vz, UU, H, x_range, z_range)

    # Print progress
    print(f"Completed processing for tp = {tp:.2f}")

# End of processing
print("All time points processed and data saved.")


In [ ]:
# Step 2: Finding Out Stagnation Points in 2D profile
import os
from pathlib import Path
import numpy as np
import pandas as pd
import h5py
import plotly.graph_objects as go
import plotly.io as pio
from scipy.interpolate import griddata
from scipy.ndimage import minimum_filter
from concurrent.futures import ThreadPoolExecutor

# Get the current working directory
script_dir = Path.cwd()

# Define the range boundaries
x_min, x_max = 0, 10000
z_min, z_max = -2000, -40
threshold = 1e-2

# Define a function to filter points that are at least 200 meters apart
def filter_points(points):
    filtered = []
    for point in points:
        if all(np.linalg.norm(np.array(point) - np.array(p)) >= 200 for p in filtered):
            filtered.append(point)
    return filtered

def process_xz_domain(X_filtered, Z_filtered, vx_filtered, vz_filtered, UU_filtered):
    # Create a grid for interpolation
    grid_x, grid_z = np.mgrid[x_min:x_max:800j, z_min:z_max:800j]  # Define a finer grid for interpolation

    # Perform interpolation for vx, vz, UU using griddata
    grid_vx = griddata((X_filtered, Z_filtered), vx_filtered, (grid_x, grid_z), method='linear')
    grid_vz = griddata((X_filtered, Z_filtered), vz_filtered, (grid_x, grid_z), method='linear')
    grid_UU = griddata((X_filtered, Z_filtered), UU_filtered, (grid_x, grid_z), method='linear')

    # Identify local minima in UU using a minimum filter
    UU_local_min = (grid_UU == minimum_filter(grid_UU, size=50))  # Find local minima in UU
    UU_local_min_points = []

    # Collect points that meet the threshold condition (UU = 0 or very close to 0)
    for i in range(grid_x.shape[0]):
        for j in range(grid_x.shape[1]):
            if UU_local_min[i, j] and x_min + 1 <= grid_x[i, j] <= x_max - 1 and z_min + 1 <= grid_z[i, j] <= z_max - 1:
                if grid_UU[i, j] < threshold:
                    UU_local_min_points.append((grid_x[i, j], grid_z[i, j], grid_UU[i, j]))

    return filter_points(UU_local_min_points)

# Aggregate and filter points from all slices
def compare_and_filter_points(points):
    filtered = []
    for point in points:
        existing_point = next((p for p in filtered if np.linalg.norm(np.array(point[:2]) - np.array(p[:2])) < 1), None)
        if existing_point:
            if point[2] < existing_point[2]:
                filtered.remove(existing_point)
                filtered.append(point)
        else:
            filtered.append(point)
    return filtered

# Save points to a CSV file
def save_points_to_csv(points, file_name, header):
    csv_path = script_dir / 'results' / file_name
    np.savetxt(csv_path, points, delimiter=', ', header=header, comments='', fmt='%s')
    print(f"Found and saved {len(points)} points to {file_name}")

# Process a CSV file and generate a Tecplot .dat file
def process_csv_file(file_prefix, title="Stagnation Points"):
    """
    Process a CSV file and generate a Tecplot .dat file.

    :param file_prefix: The prefix of the CSV file (without the .csv extension).
    :param title: The title for the ZONE in the Tecplot .dat file.
    """
    input_path = script_dir / 'results' / f'{file_prefix}.csv'
    df = pd.read_csv(input_path)

    # Remove any spaces in column names
    df.columns = df.columns.str.strip()

    # Ensure required columns are present
    required_columns = ['x', 'z']
    if 'tp' in df.columns:
        required_columns.append('tp')
    actual_columns = df.columns.tolist()

    if not all(col in actual_columns for col in required_columns):
        print(f"Error: CSV file is missing required columns. Actual columns: {actual_columns}")
        return

    # Save the .dat file with required formatting
    output_path_dat = script_dir / 'results' / f'{file_prefix}.dat'
    with open(output_path_dat, 'w') as f:
        # Write the VARIABLES declaration
        if 'tp' in df.columns:
            f.write('VARIABLES = "tp", "X", "Z", "UU"\n')
        else:
            f.write('VARIABLES = "X", "Z", "UU"\n')

        # Write the ZONE header
        num_points = len(df)
        f.write(f'ZONE T="{title}", I={num_points}, J=1, K=1, F=POINT\n')

        # Write the data points
        for index, row in df.iterrows():
            if 'tp' in df.columns:
                f.write(f"{row['tp']}\t{row['x']}\t{row['z']}\t{row['UU']}\n")
            else:
                f.write(f"{row['x']}\t{row['z']}\t{row['UU']}\n")

    print(f'File generated: {output_path_dat}')

# Generate a 2D scatter plot for visualization
def plot_points(all_points, title):
    fig = go.Figure()
    base_colors = ['#FF0000', '#FFA500', '#008000', '#87CEEB', '#00FFFF', '#0000FF', '#FF00CC', '#A52A2A', '#808080', '#000000']
    
    # Extend the color list if necessary
    num_colors_needed = len(all_points)
    colors = (base_colors * (num_colors_needed // len(base_colors) + 1))[:num_colors_needed]

    for tp, color in zip(sorted(all_points.keys()), colors):
        points = all_points[tp]
        if len(points) == 0:  # Check if points list is empty
            print(f"No points found for tp={tp:.2f}, skipping...")
            continue  # Skip this iteration if no points are found
        x, z, _ = zip(*points)
        fig.add_trace(go.Scatter(
            x=x,
            y=z,
            mode='markers',
            marker=dict(size=5, color=color),
            name=f'tp={tp}'
        ))

    # Update the layout of the figure
    fig.update_layout(
        title=title,
        xaxis_title='X',
        yaxis_title='Z',
        xaxis=dict(range=[0, 10000]),  # Set x-axis range from 4000 to 6000
        yaxis=dict(range=[-2000, 0])
        #template="plotly_white"
    )

    return fig

# Main function where data is loaded and processing is done
def main():
    all_points = {}
    all_stagnation_points = []  # List to collect all stagnation points across all time steps

    for tp in np.arange(0, 0.2, 0.1):
        file_relative_path = Path('results', f'head_and_velocity_tp_{tp:.2f}.h5')
        file_path = script_dir / file_relative_path

        # Check if the HDF5 file exists
        if not file_path.exists():
            print(f"Error: File '{file_path}' does not exist!")
            continue

        try:
            # Load data from the HDF5 file
            with h5py.File(file_path, 'r') as f:
                x_range = f['x_range'][:]
                z_range = f['z_range'][:]
                vx = f['vx'][:]
                vz = f['vz'][:]
                UU = f['UU'][:]
        except Exception as e:
            print(f"Error loading file: {e}")
            continue

        # Apply the filtering criteria to extract relevant data
        X, Z = np.meshgrid(x_range, z_range, indexing='ij')
        filtered_indices = (
            (X >= x_min) & (X <= x_max) &
            (Z >= z_min) & (Z <= z_max)
        )
        X_filtered = X[filtered_indices]
        Z_filtered = Z[filtered_indices]
        vx_filtered = vx[filtered_indices]
        vz_filtered = vz[filtered_indices]
        UU_filtered = UU[filtered_indices]

        # Process the entire xz domain for the given time step
        UU_local_min_points = process_xz_domain(X_filtered, Z_filtered, vx_filtered, vz_filtered, UU_filtered)

        # Filter and save the local minima points for each time step (tp)
        UU_local_min_points_filtered = compare_and_filter_points(UU_local_min_points)
        save_points_to_csv(UU_local_min_points_filtered, f'best_xz_tp_{tp:.2f}.csv', 'x, z, UU\n')
        process_csv_file(f'best_xz_tp_{tp:.2f}', title=f'Stagnation Points at tp={tp:.2f}')
        all_points[tp] = UU_local_min_points_filtered

        # Collect all stagnation points for this time step
        for point in UU_local_min_points_filtered:
            all_stagnation_points.append((tp, *point))  # Add the time point (tp) to the point tuple

    # Save all stagnation points to a single CSV file
    all_stagnation_points_df = pd.DataFrame(all_stagnation_points, columns=['tp', 'x', 'z', 'UU'])
    all_stagnation_csv_path = script_dir / 'results' / 'all_stagnation_points.csv'
    all_stagnation_points_df.to_csv(all_stagnation_csv_path, index=False)
    print(f"Saved all stagnation points to {all_stagnation_csv_path}")

    # Convert the CSV file to a Tecplot .dat file
    process_csv_file('all_stagnation_points', title='All Stagnation Points')

    # Plot the points for all time steps
    fig_best_xz = plot_points(all_points, 'Best Points from XZ Domain with Local Min UU')
    output_html_file = script_dir / 'results' / 'Stagline_Case1T_xz.html'
    pio.write_html(fig_best_xz, file=output_html_file, auto_open=False)
    fig_best_xz.show()

if __name__ == '__main__':
    main()

In [ ]:
#Step 3: Generate Critical Points around the Stagnation Point
import os
import numpy as np
import pandas as pd
import logging
from pathlib import Path
import plotly.graph_objects as go

# Set up logging to track events during processing
logging.basicConfig(filename='processing.log', level=logging.INFO, 
                    format='%(asctime)s - %(levelname)s - %(message)s')

# Function to read the CSV file containing best points (now only x, z)
def read_best_points(file_prefix):
    input_path = Path('results') / f'{file_prefix}.csv'
    if not input_path.exists() or input_path.stat().st_size == 0:
        logging.warning(f"File {input_path} does not exist or is empty.")
        return None
    df = pd.read_csv(input_path)
    # Strip spaces from column names to avoid access errors
    df.columns = df.columns.str.strip()
    return df

# Function to generate tracking points based on best points (incrementing x and z)
def generate_tracking_points(best_points, distances):
    tracking_points = {
        'left': [],
        'right': [],
        'up': [],
        'down': []
    }
    
    if len(best_points) == 0:
        logging.warning("Empty best_points array detected.")
        return tracking_points
    
    # Loop through best_points to generate tracking points
    for i in range(len(best_points) - 1):
        point = best_points[i]
        
        # Generate tracking points by adjusting x and z
        tracking_points['left'].append([point[0] - distances['horizontal'], point[1]])
        tracking_points['right'].append([point[0] + distances['horizontal'], point[1]])
        tracking_points['up'].append([point[0], point[1] + distances['vertical']])
        tracking_points['down'].append([point[0], point[1] - distances['vertical']])
    
    # Handle the last point separately to ensure all points are covered
    last_point = best_points[-1]
    tracking_points['left'].append([last_point[0] - distances['horizontal'], last_point[1]])
    tracking_points['right'].append([last_point[0] + distances['horizontal'], last_point[1]])
    tracking_points['up'].append([last_point[0], last_point[1] + distances['vertical']])
    tracking_points['down'].append([last_point[0], last_point[1] - distances['vertical']])
    
    return tracking_points

# Function to save the generated tracking points to a CSV file
def save_tracking_points(filename, tracking_points):
    all_points = []
    for direction, points in tracking_points.items():
        for point in points:
            all_points.append([direction, point[0], point[1]])  # Save direction, x, z
    
    df = pd.DataFrame(all_points, columns=['direction', 'x', 'z'])

    # Specify the path to the results folder
    results_dir = 'results'
    full_path = os.path.join(results_dir, filename)

    df.to_csv(full_path, index=False)
    logging.info(f'Tracking points saved to {full_path}.')

# Function to plot the best points and tracking points in a 2D space
def plot_points(best_points, tracking_points, filename, tp):
    fig = go.Figure()
    
    # Plot the best points (critical points)
    fig.add_trace(go.Scatter(x=best_points[:, 0], y=best_points[:, 1], 
                             mode='markers', marker=dict(size=5, color='black'), name='Critical Points'))
    
    color_dict = {
        'left': 'green',
        'right': 'magenta',
        'up': 'red',
        'down': 'blue'
    }
    
    # Plot each set of tracking points with the specified color
    for key, points in tracking_points.items():
        if len(points) == 0:
            continue
        
        points = np.array(points)
        fig.add_trace(go.Scatter(x=points[:, 0], y=points[:, 1], 
                                 mode='markers', marker=dict(size=3, color=color_dict[key]), 
                                 name=key.replace('_', ' ').title()))
    
    # Add tp value in the plot title
    fig.update_layout(
        title=f'2D Tracking Points for tp = {tp:.2f}',
        xaxis=dict(title='X Axis'),
        yaxis=dict(title='Z Axis'),
        margin=dict(l=0, r=0, b=0, t=0)
    )

    
# Ensure the results directory exists
    results_dir = 'results'
    if not os.path.exists(results_dir):
        os.makedirs(results_dir)
    
    # Specify the full path for the HTML file
    full_path = os.path.join(results_dir, filename)

    # Write the HTML file
    #fig.show()
    fig.write_html(str(full_path))
    logging.info(f'Plot saved as HTML: {full_path} with tp = {tp:.2f}')

# Function to process the CSV file and generate DAT and MCR files
def process_csv_file(file_prefix):
    input_path = Path('results') / f'{file_prefix}_tracking_points.csv'
    try:
        df = pd.read_csv(input_path)
        # Strip spaces from column names to avoid access errors
        df.columns = df.columns.str.strip()
    except FileNotFoundError:
        logging.error(f"File not found: {input_path}")
        return
    
    output_path_dat = Path('results') / f'{file_prefix}_tracking_points.dat'
    df[['x', 'z']].to_csv(output_path_dat, sep='\t', index=False, header=False)
    
    # Add header information to the DAT file
    with open(output_path_dat, 'r+') as f:
        content = f.read()
        f.seek(0, 0)
        f.write('VARIABLES = "X", "Z"\n')
        f.write(f"ZONE T='{file_prefix}'\n")
        f.write(f'I={len(df)}, J=1, K=1, F=POINT\n')
        f.write(content)
    
    # Generate the corresponding macro file for visualization
    generate_macro_file(df, file_prefix)
    logging.info(f'Generated files: {output_path_dat}, {file_prefix}_tracking_points.mcr')

# Function to generate a macro file for visualization
def generate_macro_file(df, file_prefix):
    macro_file_path = Path('results') / f'{file_prefix}_tracking_points.mcr'
    points = df[['x', 'z']].values.tolist()
    
    with open(macro_file_path, mode='w') as macrofile:
        macrofile.write('#!MC 1410\n')
        macrofile.write(f'$!StreamAttributes Color = blue\n')
        for point in points:
            macrofile.write('$!Streamtrace Add\n')
            macrofile.write('  StreamType = TwoDLine\n')
            macrofile.write('  StreamDirection = Both\n')
            macrofile.write('  StartPos\n')
            macrofile.write('    {\n')
            macrofile.write(f'    X = {point[0]}\n')
            macrofile.write(f'    Y = {point[1]}\n')
            macrofile.write('    }\n')
        macrofile.write('$!REDRAWALL\n')
    logging.info(f'Macro file saved: {macro_file_path}')

# Main function to process all files and generate outputs
def main():
    result_folder = Path(os.getcwd()) / 'results'
    tracking_distances = {
        'horizontal': 50,  # Horizontal (left and right) distance
        'vertical': 25     # Vertical (up and down) distance
    }

    # Loop through different tp values to process each corresponding file
    for tp in np.arange(0, 1, 0.25):
        # Update file prefix as per your new requirement
        file_prefix = f'best_xz_tp_{tp:.2f}'
        
        best_points_df = read_best_points(file_prefix)
        if best_points_df is None:
            continue
        
        # Only keep x and z (strip spaces to avoid issues with column names)
        best_points = best_points_df[['x', 'z']].values  

        # Generate tracking points for the current tp
        tracking_points = generate_tracking_points(best_points, tracking_distances)
        
        # Save the generated tracking points
        save_tracking_points(f'{file_prefix}_tracking_points.csv', tracking_points)

        # Plot the points and save to an HTML file
        plot_points(best_points, tracking_points, Path(f'{file_prefix}_tracking_points.html'), tp)

        # Process CSV and generate corresponding DAT and MCR files
        process_csv_file(file_prefix)
        
print("Code execution completed successfully.")

if __name__ == '__main__':
    main()


In [ ]:
#Step 4: Generate Dividing Streamlines across Critial Points at different tp
import numpy as np
import h5py
import os
import pandas as pd
import plotly.graph_objects as go
from scipy.integrate import solve_ivp
from concurrent.futures import ThreadPoolExecutor
from numba import jit
import plotly.io as pio

# Function to read data from an HDF5 file (only x and z components of velocity)
def read_data_hdf5(filename):
    with h5py.File(filename, 'r') as f:
        vx = f['vx'][:]  # Velocity component in x-direction
        vz = f['vz'][:]  # Velocity component in z-direction
        x_range = f['x_range'][:]  # Range of x values
        z_range = f['z_range'][:]  # Range of z values
    return vx, vz, x_range, z_range

# JIT-compiled function to compute the velocity field at a given position
@jit(nopython=True)
def velocity_field_numba(pos, vx, vz, x_range, z_range):
    x, z = pos
    # Check if the position is outside the defined ranges
    if x < x_range[0] or x > x_range[-1] or z < z_range[0] or z > z_range[-1]:
        return np.array([0.0, 0.0])  # Return zero velocity if out of bounds
    
    # Find the indices of the grid cell containing the position
    xi = np.searchsorted(x_range, x) - 1
    zi = np.searchsorted(z_range, z) - 1

    # Get the surrounding grid points
    x1, x2 = x_range[xi], x_range[xi+1]
    z1, z2 = z_range[zi], z_range[zi+1]

    # Compute relative distances within the grid cell
    xd = (x - x1) / (x2 - x1)
    zd = (z - z1) / (z2 - z1)

    # Trilinear interpolation for the x-component of velocity
    vx_val = vx[xi, zi] * (1 - xd) + vx[xi + 1, zi] * xd

    # Trilinear interpolation for the z-component of velocity
    vz_val = vz[xi, zi] * (1 - xd) + vz[xi + 1, zi] * xd

    return np.array([vx_val, vz_val])  # Return the interpolated velocity vector

# Function to compute streamlines based on velocity fields and start points
def compute_streamlines(vx, vz, x_range, z_range, start_points, max_distance=1e7, tol=1e-7):
    # Define the velocity field function for the ODE solver
    def velocity_field(t, pos):
        return velocity_field_numba(pos, vx, vz, x_range, z_range)

    # Define an event to terminate integration when a boundary condition is met
    def boundary_event(t, pos):
        return pos[1] + 60  # Termination condition, e.g., z = -80 surface

    boundary_event.terminal = True  # Stop integration when event is triggered
    boundary_event.direction = 0     # Event is detected regardless of direction

    # Function to integrate the streamline starting from a given point
    def integrate_streamline(start):
        if start[1] > -60:  # Skip if the start point is above the boundary
            return np.array([])

        # Integrate forward in time
        result_forward = solve_ivp(velocity_field, [0, max_distance], start, method='RK45', rtol=tol, atol=tol, events=boundary_event)
        
        # Integrate backward in time
        result_backward = solve_ivp(velocity_field, [0, -max_distance], start, method='RK45', rtol=tol, atol=tol, events=boundary_event)
        
        # Extract the streamline points
        streamline_forward = result_forward.y.T
        streamline_backward = result_backward.y.T
        
        # Reverse the backward streamline and combine with forward streamline
        streamline_backward = streamline_backward[::-1]
        streamline = np.vstack((streamline_backward, streamline_forward))
        
        return streamline

    # Use a thread pool to parallelize streamline computations
    with ThreadPoolExecutor() as executor:
        streamlines = list(executor.map(integrate_streamline, start_points))
    
    return streamlines  # Return the list of computed streamlines

# Function to plot streamlines and their corresponding start points
def plot_streamlines_and_points(fig, streamlines, start_points, directions, colors):
    for streamline, start, direction in zip(streamlines, start_points, directions):
        if len(streamline) > 0:
            color = colors[directions.index(direction)]
            fig.add_trace(go.Scatter(
                x=streamline[:, 0], y=streamline[:, 1],
                mode='lines',
                line=dict(color=color, width=4)  # Set streamline color and width
            ))
    
    # Plot the start points of the streamlines
    for start, direction in zip(start_points, directions):
        color = colors[directions.index(direction)]
        fig.add_trace(go.Scatter(
            x=[start[0]], y=[start[1]],
            mode='markers',
            marker=dict(color=color, size=5, symbol='square'),  # Set marker color, size, and shape
            name=f'Start Point {direction}'
        ))

# Function to read trace points from a CSV file
def read_trace_points(file_path):
    # Read the CSV file
    df = pd.read_csv(file_path, names=['direction', 'x', 'z'], header=0)
    
    directions = df['direction'].tolist()
    start_points = df[['x', 'z']].values  # Return only the x, z coordinates
    
    return directions, start_points

# Get the current working directory as the base path for the script
current_working_dir = os.getcwd()

# Define the path to the results folder and create it if it doesn't exist
result_folder = os.path.join(current_working_dir, 'results')
os.makedirs(result_folder, exist_ok=True)

# Define a list of colors to be used for different tracking point categories
colors = ['magenta', 'green', 'red', 'blue']

# Define prefixes for the tracking point files
tracking_point_prefixes = [
    'best_xz'
]

# Loop through different tp values to process each corresponding file
for tp in np.arange(0, 1, 0.25):   # Iterate over tp from 0 to 1 with step 0.25
    # Define the path to the input HDF5 file for the current tp value
    input_path_hdf5 = os.path.join(result_folder, f'head_and_velocity_tp_{tp:.2f}.h5')
    
    # Read data from the HDF5 file
    vx, vz, x_range, z_range = read_data_hdf5(input_path_hdf5)

    # Create a new Plotly figure for visualization
    fig = go.Figure()

    # Loop through each tracking point category and its corresponding color
    for prefix in tracking_point_prefixes:
        # Define the path to the tracking point CSV file
        file_path = os.path.join(result_folder, f'{prefix}_tp_{tp:.2f}_tracking_points.csv')
        # Read the start points and directions for streamlines
        directions, start_points = read_trace_points(file_path)
        
        # Compute streamlines based on the velocity field and start points
        streamlines = compute_streamlines(vx, vz, x_range, z_range, start_points)
        
        # Plot the computed streamlines and their start points on the figure
        plot_streamlines_and_points(fig, streamlines, start_points, directions, colors)

    # Update the layout of the plot to set axis ranges
    fig.update_layout(
        xaxis=dict(title='X', range=[x_range[0], x_range[-1]]),
        yaxis=dict(title='Z', range=[z_range[0], z_range[-1]]),
        title=f'Streamlines at tp = {tp:.2f}',
        showlegend=True
    )

    # Save the plot as an HTML file and display it
    html_file_path = os.path.join(result_folder, f'streamlines_tp_{tp:.2f}.html')
    fig.write_html(html_file_path)
    fig.show()  # Show the plot in the browser

In [ ]:
# Step 5-1: Generate Dividing Streamlines under the average flow field
import numpy as np
import h5py
import os
from scipy.ndimage import gaussian_filter
from scipy.optimize import minimize
from scipy.integrate import solve_ivp
from concurrent.futures import ThreadPoolExecutor
import plotly.graph_objects as go
import pandas as pd

# Function to compute the average hydraulic head and velocity components
def compute_average_flow(input_folder, output_filename):
    """
    Computes the average hydraulic head and velocity components from multiple time-step HDF5 files.
    """
    H_sum, vx_sum, vz_sum, count = None, None, None, 0

    for tp in np.arange(0, 1, 0.1):
        hdf5_filename = os.path.join(input_folder, 'head_and_velocity_tp_' + '{:.2f}'.format(tp) + '.h5')
        if not os.path.exists(hdf5_filename):
            print("File not found: " + hdf5_filename + ". Skipping.")
            continue

        try:
            with h5py.File(hdf5_filename, 'r') as f:
                H = f['H'][:]
                vx = f['vx'][:]
                vz = f['vz'][:]
                x_range = f['x_range'][:]
                z_range = f['z_range'][:]
        except OSError as e:
            print("Error reading file " + hdf5_filename + ": " + str(e))
            continue

        if H_sum is None:
            H_sum = np.zeros_like(H)
            vx_sum = np.zeros_like(vx)
            vz_sum = np.zeros_like(vz)

        H_sum += H
        vx_sum += vx
        vz_sum += vz
        count += 1

    if count == 0:
        print("No files processed. Exiting.")
        return None

    Have = H_sum / count
    Vxave = vx_sum / count
    Vzave = vz_sum / count
    UUave = np.sqrt(Vxave**2 + Vzave**2)

    try:
        with h5py.File(output_filename, 'w') as f:
            f.create_dataset('Have', data=Have)
            f.create_dataset('Vxave', data=Vxave)
            f.create_dataset('Vzave', data=Vzave)
            f.create_dataset('UUave', data=UUave)
            f.create_dataset('x_range', data=x_range)
            f.create_dataset('z_range', data=z_range)
    except OSError as e:
        print("Error writing output file " + output_filename + ": " + str(e))
        return None

    
# Save the average flow field to a Tecplot .dat file
    dat_filename = os.path.splitext(output_filename)[0] + '.dat'
    with open(dat_filename, 'w') as f:
        f.write('TITLE = "Average Flow Field"\n')
        f.write('VARIABLES = "X", "Z", "Hydraulic Head (H)", "Velocity X (Vx)", "Velocity Z (Vz)", "Velocity Magnitude (UU)"\n')

        # Find the indices where z < -20
        z_indices = np.where(z_range < -20)[0]

        # If there are no indices where z < -20, exit the function
        if len(z_indices) == 0:
            print("No data found for z < -20. Exiting.")
            return Have, Vxave, Vzave, UUave, x_range, z_range

        # Write the ZONE header with the new J size
        f.write('ZONE T="Average Flow Field", I={}, J={}, F=POINT\n'.format(Have.shape[1], len(z_indices)))

        for i in z_indices:
            for j in range(Have.shape[1]):
                f.write('{:.6e} {:.6e} {:.6e} {:.6e} {:.6e} {:.6e}\n'.format(
                    x_range[j], z_range[i],
                    Have[j, i], Vxave[j, i], Vzave[j, i], UUave[j, i]
                ))

    print(f"Average flow field has been saved to {output_filename} and {dat_filename}.")

    return Have, Vxave, Vzave, UUave, x_range, z_range


# Function to find stagnation points
def find_stagnation_points(vx, vz, x_range, z_range):
    """
    Identifies stagnation points based on the velocity magnitude minimum.
    """
    uu = np.sqrt(vx**2 + vz**2)
    uu_smoothed = gaussian_filter(uu, sigma=2)

    stagnation_points = []
    for i in range(1, uu.shape[0] - 1):
        for j in range(1, uu.shape[1] - 1):
            if uu_smoothed[i, j] < uu_smoothed[i-1, j] and uu_smoothed[i, j] < uu_smoothed[i+1, j] and \
               uu_smoothed[i, j] < uu_smoothed[i, j-1] and uu_smoothed[i, j] < uu_smoothed[i, j+1]:
                x = x_range[i]
                z = z_range[j]
                result = minimize(
                    lambda pos: np.sqrt(vx[int(pos[0]), int(pos[1])]**2 + vz[int(pos[0]), int(pos[1])]**2),
                    x0=[i, j],
                    bounds=((0, len(x_range)-1), (0, len(z_range)-1))
                )
                if int(result.x[0]) >= 0 and int(result.x[1]) >= 0:
                    stagnation_points.append((x_range[int(result.x[0])], z_range[int(result.x[1])]))

    return stagnation_points

# Function to generate critical points around a stagnation point
def generate_critical_points(stagnation_point, dx, dz):
    """
    Generate critical points around a given stagnation point with direction information.
    """
    x, z = stagnation_point
    return [
        ((x - dx, z), 'left'),  # Left
        ((x + dx, z), 'right'), # Right
        ((x, z + dz), 'up'),    # Up
        ((x, z - dz), 'down')   # Down
    ]

# Interpolation function for velocity
def interpolate_velocity(x, z, x_range, z_range, vx, vz):
    """
    Performs bilinear interpolation to estimate velocity components at a given point.
    """
    if x < x_range[0] or x > x_range[-1] or z < z_range[0] or z > z_range[-1]:
        return [0, 0]
    xi = np.searchsorted(x_range, x) - 1
    zi = np.searchsorted(z_range, z) - 1
    x1, x2 = x_range[xi], x_range[xi + 1]
    z1, z2 = z_range[zi], z_range[zi + 1]
    xd = (x - x1) / (x2 - x1)
    zd = (z - z1) / (z2 - z1)

    vx_val = (vx[xi, zi] * (1 - xd) * (1 - zd) +
              vx[xi + 1, zi] * xd * (1 - zd) +
              vx[xi, zi + 1] * (1 - xd) * zd +
              vx[xi + 1, zi + 1] * xd * zd)

    vz_val = (vz[xi, zi] * (1 - xd) * (1 - zd) +
              vz[xi + 1, zi] * xd * (1 - zd) +
              vz[xi, zi + 1] * (1 - xd) * zd +
              vz[xi + 1, zi + 1] * xd * zd)

    return [vx_val, vz_val]

# Compute streamlines for given starting points
def compute_streamlines(vx, vz, x_range, z_range, start_points, max_distance=1e7, tol=1e-7):
    def velocity_field(t, pos):
        x, z = pos
        if x < x_range[0] or x > x_range[-1] or z < z_range[0] or z > z_range[-1]:
            return [0, 0]
        
        xi = np.searchsorted(x_range, x) - 1
        zi = np.searchsorted(z_range, z) - 1
        x1, x2 = x_range[xi], x_range[xi + 1]
        z1, z2 = z_range[zi], z_range[zi + 1]
        xd = (x - x1) / (x2 - x1)
        zd = (z - z1) / (z2 - z1)

        vx_val = (vx[xi, zi] * (1 - xd) * (1 - zd) +
                  vx[xi + 1, zi] * xd * (1 - zd) +
                  vx[xi, zi + 1] * (1 - xd) * zd +
                  vx[xi + 1, zi + 1] * xd * zd)

        vz_val = (vz[xi, zi] * (1 - xd) * (1 - zd) +
                  vz[xi + 1, zi] * xd * (1 - zd) +
                  vz[xi, zi + 1] * (1 - xd) * zd +
                  vz[xi + 1, zi + 1] * xd * zd)

        return [vx_val, vz_val]

    # Define an event to terminate integration when a boundary condition is met
    def boundary_event(t, pos):
        x, z = pos
        return pos[1] + 40  # Termination condition, e.g., z = -80 surface

    boundary_event.terminal = True  # Stop integration when event is triggered
    boundary_event.direction = 0     # Event is detected regardless of direction    

    def integrate_streamline(start):
        if np.ndim(start) != 1:
            start = np.array(start).flatten()

        result_forward = solve_ivp(
            velocity_field, 
            [0, max_distance], 
            start, 
            method='RK45', 
            rtol=tol, 
            atol=tol, 
            events=boundary_event
        )

        result_backward = solve_ivp(
            velocity_field, 
            [0, -max_distance], 
            start, 
            method='RK45', 
            rtol=tol, 
            atol=tol, 
            events=boundary_event
        )

        streamline_forward = result_forward.y.T
        streamline_backward = result_backward.y.T[::-1]
        streamline = np.vstack((streamline_backward, streamline_forward))
        return streamline

    if isinstance(start_points, list):
        start_points = [np.array(start).flatten() for start in start_points]

    with ThreadPoolExecutor() as executor:
        streamlines = list(executor.map(integrate_streamline, start_points))

    return streamlines

# Function to plot dividing streamlines across critical points around stagnation points
def plot_streamlines(fig, streamlines, stagnation_points, colors):
    for streamline, color in zip(streamlines, colors):
        fig.add_trace(go.Scatter(x=streamline[:, 0], y=streamline[:, 1], mode='lines', line=dict(width=2, color=color)))
    for x, z in stagnation_points:
        fig.add_trace(go.Scatter(x=[x], y=[z], mode='markers', marker=dict(color='black', size=10)))

# Main workflow
input_folder = os.path.join(os.getcwd(), 'results')
output_filename = './results/average_flow.h5'
Have, Vxave, Vzave, UUave, x_range, z_range = compute_average_flow(input_folder, output_filename)

stagnation_points = find_stagnation_points(Vxave, Vzave, x_range, z_range)
print(f"Stagnation points found: {stagnation_points}")

# Prepare colors for each direction
colors = {
    'left': 'magenta',
    'right': 'green',
    'up': 'red',
    'down': 'blue'
}

# Generate critical points with directions
critical_points_with_directions = [point for sp in stagnation_points for point in generate_critical_points(sp, 200, 200)]

# Separate critical points by their direction for streamline computation
start_points_by_direction = {}
for point, direction in critical_points_with_directions:
    if direction not in start_points_by_direction:
        start_points_by_direction[direction] = []
    start_points_by_direction[direction].append(point)

# Compute and plot streamlines for each direction separately
fig = go.Figure()
all_streamlines = []  # Store all streamlines for exporting to CSV

for direction, start_points in start_points_by_direction.items():
    streamlines = compute_streamlines(Vxave, Vzave, x_range, z_range, start_points)
    plot_streamlines(fig, streamlines, stagnation_points, [colors[direction]] * len(streamlines))
    all_streamlines.extend(streamlines)

# Save the streamlines to a CSV file
result_folder = os.path.join(os.getcwd(), 'results')
os.makedirs(result_folder, exist_ok=True)
csv_filename = os.path.join(result_folder, 'streamlines.csv')

streamline_data = []
for i, streamline in enumerate(all_streamlines):
    for point in streamline:
        streamline_data.append([i, point[0], point[1]])

df = pd.DataFrame(streamline_data, columns=['streamline_id', 'x', 'z'])
df.to_csv(csv_filename, index=False)

# Save the plot as an HTML file
html_filename = os.path.join(result_folder, 'streamlines.html')
fig.write_html(html_filename)
print(f"Plot has been saved to {html_filename}.")

# Show the plot
fig.show()

In [ ]:
# Step 5-2: Generate Streamlines across specified points under the average flow field
import os
import h5py
import numpy as np
from scipy.integrate import solve_ivp
from concurrent.futures import ThreadPoolExecutor
import plotly.graph_objects as go
import pandas as pd

# Define the fixed tracking points
tracking_points = [
    (4000, -310), (4200, -310),(4400, -310),(4600, -310),(4800, -310),
    (5200, -310), (5400, -310),(5600, -310),(5800, -310),(6000, -310),
    (5000, -510), (5000, -710),(5000, -110)
]

# Load the average flow field from the HDF5 file
def load_average_flow(h5_filename):
    with h5py.File(h5_filename, 'r') as f:
        Have = f['Have'][:]
        Vxave = f['Vxave'][:]
        Vzave = f['Vzave'][:]
        x_range = f['x_range'][:]
        z_range = f['z_range'][:]
    return Have, Vxave, Vzave, x_range, z_range

# Compute streamlines for given starting points
def compute_streamline(vx, vz, x_range, z_range, start_point, max_distance=1e7, tol=1e-7):
    def velocity_field(t, pos):
        x, z = pos
        if x < x_range[0] or x > x_range[-1] or z < z_range[0] or z > z_range[-1]:
            return [0, 0]
        
        xi = np.searchsorted(x_range, x) - 1
        zi = np.searchsorted(z_range, z) - 1
        x1, x2 = x_range[xi], x_range[xi + 1]
        z1, z2 = z_range[zi], z_range[zi + 1]
        xd = (x - x1) / (x2 - x1)
        zd = (z - z1) / (z2 - z1)

        vx_val = (vx[xi, zi] * (1 - xd) * (1 - zd) +
                  vx[xi + 1, zi] * xd * (1 - zd) +
                  vx[xi, zi + 1] * (1 - xd) * zd +
                  vx[xi + 1, zi + 1] * xd * zd)

        vz_val = (vz[xi, zi] * (1 - xd) * (1 - zd) +
                  vz[xi + 1, zi] * xd * (1 - zd) +
                  vz[xi, zi + 1] * (1 - xd) * zd +
                  vz[xi + 1, zi + 1] * xd * zd)

        return [vx_val, vz_val]

    # Define an event to terminate integration when a boundary condition is met
    def boundary_event(t, pos):
        x, z = pos
        return pos[1] + 40  # Termination condition, e.g., z = -80 surface

    boundary_event.terminal = True  # Stop integration when event is triggered
    boundary_event.direction = 0     # Event is detected regardless of direction    

    def integrate_streamline(start, direction):
        result = solve_ivp(
            velocity_field, 
            [0, direction * max_distance], 
            start, 
            method='RK45', 
            rtol=tol, 
            atol=tol, 
            events=boundary_event
        )
        streamline = result.y.T
        return streamline

    forward = integrate_streamline(start_point, 1)
    backward = integrate_streamline(start_point, -1)
    
    return forward, backward

# Main workflow
h5_filename = './results/average_flow.h5'
output_folder = './results'

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

Have, Vxave, Vzave, x_range, z_range = load_average_flow(h5_filename)

fig = go.Figure()

all_streamlines = []  # Store all streamlines for exporting to CSV

with ThreadPoolExecutor() as executor:
    futures = {executor.submit(compute_streamline, Vxave, Vzave, x_range, z_range, point): point for point in tracking_points}
    for future in futures:
        forward, backward = future.result()
        fig.add_trace(go.Scatter(x=forward[:, 0], y=forward[:, 1], mode='lines', line=dict(width=2, color='blue')))
        fig.add_trace(go.Scatter(x=backward[:, 0], y=backward[:, 1], mode='lines', line=dict(width=2, color='red')))
        all_streamlines.append(forward)
        all_streamlines.append(backward)

# Save the streamlines to a CSV file
csv_filename = os.path.join(output_folder, 'tracked_streamlines.csv')

streamline_data = []
for i, streamline in enumerate(all_streamlines):
    for point in streamline:
        streamline_data.append([i, point[0], point[1]])

df = pd.DataFrame(streamline_data, columns=['streamline_id', 'x', 'z'])
df.to_csv(csv_filename, index=False)

# Save the plot as an HTML file
html_filename = os.path.join(output_folder, 'tracked_streamlines.html')
fig.write_html(html_filename)
print(f"Plot has been saved to {csv_filename} and {html_filename}.")

# Show the plot
fig.show()

In [ ]:
#Step 6-1:Generate particles trajectories across specified points, accompanied by streamlines under average field. 
#Particles are released from t=0P
import numpy as np
import os
import pandas as pd
from numba import jit, prange
import plotly.graph_objects as go

# Parameter settings (ensure these are already defined)
params = {
    'p': 3, 'q': 3, 'HRx': 6, 'HLx': 3, 'D': -2000, 'a': 2/3, 'b': 1/3,
    'LX': 10000, 'LY': 10000, 'TauD': 0.4
}

# Extract parameters
p, q, HRx, HLx, D, a, b, TauD = [params[k] for k in ('p', 'q', 'HRx', 'HLx', 'D', 'a', 'b', 'TauD')]
LX, LY = params['LX'], params['LY']

# Precompute mesh grid and cosine values
mesh_x, mesh_z = 101, 101
x_range = np.linspace(0, LX, mesh_x)
z_range = np.linspace(D, 0, mesh_z)
b1 = np.cos(np.pi * x_range / LX)
b2 = np.cos(p * np.pi * x_range / LX)

# Precompute cosh values for efficiency
c1 = np.cosh(np.pi * D / LX)
c2 = np.cosh(p * np.pi * D / LX)

@jit(nopython=True, parallel=True)
def compute_J(e1, e2):
    """Calculate the J matrix and related variables in 2D (x, z) space."""
    J = np.zeros((mesh_x, mesh_z))
    for i in prange(mesh_x):
        for k in prange(mesh_z):
            hA = HRx - HRx * b1[i] * np.cosh(np.pi * (z_range[k] - D) / LX) / c1
            hB = HLx - HLx * b2[i] * np.cosh(p * np.pi * (z_range[k] - D) / LX) / c2
            g1, g2 = 0.0, 0.0
            for n in range(1, 1000):  
                k1, an = (-1) ** (n - 1), (2 * n - 1) * np.pi / 2
                b0 = np.cos(an * (z_range[k] - D) / D)
                f1 = D * (k1 * an * b0 * (1 / an**2 + (an**2 * e1 + 2 * np.pi * TauD * e2) / (an**4 + 4 * np.pi**2 * TauD**2)))
                g1 += f1
                k2, d1 = (-1) ** n, an**2 + (p * np.pi * D / LX)**2
                f2 = D * (k2 * an * b0 * (1 / d1 + (d1 * e1 + 2 * np.pi * TauD * e2) / (d1**2 + 4 * np.pi**2 * TauD**2)))
                g2 += f2               
            hC = HLx * g1 / D + HLx * b2[i] * g2 / D
            J[i, k] = hA + a * hB + b * hC
    return J

def calculate_fields(tp):
    e1 = np.cos(2 * np.pi * tp)
    e2 = np.sin(2 * np.pi * tp)
    J = compute_J(e1, e2)
    dx, dz = x_range[1] - x_range[0], z_range[1] - z_range[0]
    grad_x, grad_z = np.gradient(J, dx, dz)
    vx, vz = -100000 * grad_x, -100000 * grad_z
    H = J  # Hydraulic head (H) is equivalent to J
    return H, vx, vz  # Removed UU from the return statement

# Function to get velocity components at a specific point (x, z) without recalculating the entire field each time
def get_velocity_components(vx, vz, x, z):
    """Retrieve the velocity components at a specific point (x, z)."""
    ix = np.abs(x_range - x).argmin()
    iz = np.abs(z_range - z).argmin()
    return vx[ix, iz], vz[ix, iz]

def track_particles(tracking_points, tp_start, tp_end, step_size, x_range=(0, 10000), z_range=(-2000, -40)):
    positions = []
    # Precompute the velocity fields for all time points once and store them in a dictionary
    if step_size > 0:
        time_points = np.round(np.arange(tp_start, tp_end + step_size, step_size), decimals=2)
    else:
        time_points = np.round(np.arange(tp_start, tp_end - step_size, step_size), decimals=2)  # For backward tracking

    velocity_fields = {tp: calculate_fields(tp)[1:] for tp in time_points}
    
    for x_start, z_start in tracking_points:
        x, z = x_start, z_start
        traj = [(x, z, tp_start)]
        
        if step_size > 0:
            iter_time_points = time_points[time_points > tp_start]
        else:
            iter_time_points = time_points[time_points < tp_start]  # For backward tracking
        
        for tp in iter_time_points:
            try:
                vx, vz = velocity_fields[tp]
            except KeyError as e:
                print(f"KeyError encountered: {e}. Available keys: {list(velocity_fields.keys())}")
                raise
            vx_at_point, vz_at_point = get_velocity_components(vx, vz, x, z)
            x_new, z_new = x + vx_at_point * step_size, z + vz_at_point * step_size
            
            if not (x_range[0] <= x_new <= x_range[1] and z_range[0] <= z_new <= z_range[1]):
                break
            
            traj.append((x_new, z_new, tp))
            x, z = x_new, z_new
        
        positions.append(np.array(traj))
    
    return positions

# Set initial particle position and time parameters
tp_start, tp_end, step_size = 0, 50, 0.1
tp_back_start, tp_back_end, step_back_size = 0, -50, -0.1

# Define the fixed tracking points
tracking_points = [
    (4000, -310), (4200, -310),(4400, -310),(4600, -310),(4800, -310),
    (5200, -310), (5400, -310),(5600, -310),(5800, -310),(6000, -310),
    (5000, -510), (5000, -710),(5000, -110)]

# Track particles for the specified tracking points using the optimized function
all_trajectories = track_particles(tracking_points, tp_start, tp_end, step_size)
all_backward_trajectories = track_particles(tracking_points, tp_back_start, tp_back_end, step_back_size)
    
# Function to load tracked streamlines from CSV
def load_tracked_streamlines(csv_filename):
    df = pd.read_csv(csv_filename)
    streamlines = []
    for streamline_id in df['streamline_id'].unique():
        streamline_data = df[df['streamline_id'] == streamline_id][['x', 'z']].values
        streamlines.append(streamline_data)
    return streamlines

# Load tracked streamlines from CSV
current_working_dir = os.getcwd()
result_folder = os.path.join(current_working_dir, 'results')
csv_filename = os.path.join(result_folder, 'tracked_streamlines.csv')
streamlines = load_tracked_streamlines(csv_filename)

# Plotting setup
fig = go.Figure()

# Color scales for forward and backward trajectories
colorscale_forward = [[0, 'rgba(0, 255, 0, 1)'], [0.5, 'rgba(255, 255, 0, 0.5)'], [1, 'rgba(255, 0, 0, 1)']]
colorscale_backward = [[0, 'rgba(0, 0, 255, 1)'], [0.5, 'rgba(0, 255, 255, 0.5)'], [1, 'rgba(0, 255, 0, 1)']]

# Add traces for forward and backward trajectories
for traj, colorscale in zip([all_trajectories, all_backward_trajectories], [colorscale_forward, colorscale_backward]):
    for path in traj:
        fig.add_trace(go.Scatter(
            x=path[:, 0], 
            y=path[:, 1], 
            mode='lines+markers',
            marker=dict(size=5, color=path[:, 2], colorscale=colorscale, showscale=False),
            name='Forward' if traj is all_trajectories else 'Backward'
        ))

# Plot tracked streamlines
for streamline in streamlines:
    fig.add_trace(go.Scatter(
        x=streamline[:, 0], 
        y=streamline[:, 1], 
        mode='lines', 
        line=dict(width=2, color='black'),
        name='Streamline'
    ))

# Add starting points as black filled circles
start_points_x = [point[0] for point in tracking_points]
start_points_z = [point[1] for point in tracking_points]

fig.add_trace(go.Scatter(
    x=start_points_x,
    y=start_points_z,
    mode='markers',
    marker=dict(size=8, color='black', symbol='circle', line=dict(width=2, color='white')),
    name='Starting Points'
))

# Update layout
fig.update_layout(
    title="Particle Trajectories and Streamlines (Fixed Starting Points)",
    xaxis_title='X',
    yaxis_title='Z',
    margin=dict(r=10, l=10, b=10, t=40),
    xaxis=dict(range=[0, 10000]),
    yaxis=dict(range=[-2000, -40], scaleanchor="x", scaleratio=1),
    showlegend=False
)

# Custom color bar for both forward and backward trajectories
fig.update_coloraxes(
    colorscale=[[0, '#0000FF'], [0.5, '#00FF00'], [1, '#FF0000']],
    cmin=-50,
    cmax=50,
    colorbar=dict(
        title='Time (tp)',
        tickvals=[-50, -25, 0, 25, 50],
        ticktext=['-50', '-25', '0', '25', '50']
    )
)

# Show the plot
fig.show()

# Save the plot as an HTML file
html_filename = os.path.join(result_folder, 'basin_scale_particle_trajectories_with_streamlines_taud_0.4_0P.html')
fig.write_html(html_filename)

# Optionally, save the trajectory to a CSV file
#csv_filename = os.path.join(result_folder, 'basin_scale_particle_trajectories_with_streamlines_taud_0.4_0P.csv')
#np.savetxt(csv_filename, np.vstack([traj.reshape(-1, 3) for traj in all_trajectories + all_backward_trajectories]), delimiter=',', header='x,z,tp', comments='')

print(f"Particle trajectories and streamlines for fixed starting points have been calculated and saved to {html_filename} and {csv_filename}.")

In [ ]:
#Step 6-1 plus: Local enlarged image
#Generate particles trajectories across specified points, accompanied by streamlines under average field. 
#Particles are released from t=0P
import numpy as np
import os
import pandas as pd
from numba import jit, prange
import plotly.graph_objects as go

# Parameter settings (ensure these are already defined)
params = {
    'p': 3, 'q': 3, 'HRx': 6, 'HLx': 3, 'D': -2000, 'a': 2/3, 'b': 1/3,
    'LX': 10000, 'LY': 10000, 'TauD': 0.4
}

# Extract parameters
p, q, HRx, HLx, D, a, b, TauD = [params[k] for k in ('p', 'q', 'HRx', 'HLx', 'D', 'a', 'b', 'TauD')]
LX, LY = params['LX'], params['LY']

# Precompute mesh grid and cosine values
mesh_x, mesh_z = 101, 101
x_range = np.linspace(0, LX, mesh_x)
z_range = np.linspace(D, 0, mesh_z)
b1 = np.cos(np.pi * x_range / LX)
b2 = np.cos(p * np.pi * x_range / LX)

# Precompute cosh values for efficiency
c1 = np.cosh(np.pi * D / LX)
c2 = np.cosh(p * np.pi * D / LX)

@jit(nopython=True, parallel=True)
def compute_J(e1, e2):
    """Calculate the J matrix and related variables in 2D (x, z) space."""
    J = np.zeros((mesh_x, mesh_z))
    for i in prange(mesh_x):
        for k in prange(mesh_z):
            hA = HRx - HRx * b1[i] * np.cosh(np.pi * (z_range[k] - D) / LX) / c1
            hB = HLx - HLx * b2[i] * np.cosh(p * np.pi * (z_range[k] - D) / LX) / c2
            g1, g2 = 0.0, 0.0
            for n in range(1, 1000):  
                k1, an = (-1) ** (n - 1), (2 * n - 1) * np.pi / 2
                b0 = np.cos(an * (z_range[k] - D) / D)
                f1 = D * (k1 * an * b0 * (1 / an**2 + (an**2 * e1 + 2 * np.pi * TauD * e2) / (an**4 + 4 * np.pi**2 * TauD**2)))
                g1 += f1
                k2, d1 = (-1) ** n, an**2 + (p * np.pi * D / LX)**2
                f2 = D * (k2 * an * b0 * (1 / d1 + (d1 * e1 + 2 * np.pi * TauD * e2) / (d1**2 + 4 * np.pi**2 * TauD**2)))
                g2 += f2               
            hC = HLx * g1 / D + HLx * b2[i] * g2 / D
            J[i, k] = hA + a * hB + b * hC
    return J

def calculate_fields(tp):
    e1 = np.cos(2 * np.pi * tp)
    e2 = np.sin(2 * np.pi * tp)
    J = compute_J(e1, e2)
    dx, dz = x_range[1] - x_range[0], z_range[1] - z_range[0]
    grad_x, grad_z = np.gradient(J, dx, dz)
    vx, vz = -100000 * grad_x, -100000 * grad_z
    H = J  # Hydraulic head (H) is equivalent to J
    return H, vx, vz  # Removed UU from the return statement

# Function to get velocity components at a specific point (x, z) without recalculating the entire field each time
def get_velocity_components(vx, vz, x, z):
    """Retrieve the velocity components at a specific point (x, z)."""
    ix = np.abs(x_range - x).argmin()
    iz = np.abs(z_range - z).argmin()
    return vx[ix, iz], vz[ix, iz]

def track_particles(tracking_points, tp_start, tp_end, step_size, x_range=(0, 10000), z_range=(-2000, -40)):
    positions = []
    # Precompute the velocity fields for all time points once and store them in a dictionary
    if step_size > 0:
        time_points = np.round(np.arange(tp_start, tp_end + step_size, step_size), decimals=2)
    else:
        time_points = np.round(np.arange(tp_start, tp_end - step_size, step_size), decimals=2)  # For backward tracking

    velocity_fields = {tp: calculate_fields(tp)[1:] for tp in time_points}
    
    for x_start, z_start in tracking_points:
        x, z = x_start, z_start
        traj = [(x, z, tp_start)]
        
        if step_size > 0:
            iter_time_points = time_points[time_points > tp_start]
        else:
            iter_time_points = time_points[time_points < tp_start]  # For backward tracking
        
        for tp in iter_time_points:
            try:
                vx, vz = velocity_fields[tp]
            except KeyError as e:
                print(f"KeyError encountered: {e}. Available keys: {list(velocity_fields.keys())}")
                raise
            vx_at_point, vz_at_point = get_velocity_components(vx, vz, x, z)
            x_new, z_new = x + vx_at_point * step_size, z + vz_at_point * step_size
            
            if not (x_range[0] <= x_new <= x_range[1] and z_range[0] <= z_new <= z_range[1]):
                break
            
            traj.append((x_new, z_new, tp))
            x, z = x_new, z_new
        
        positions.append(np.array(traj))
    
    return positions

# Set initial particle position and time parameters
tp_start, tp_end, step_size = 0, 50, 0.1
tp_back_start, tp_back_end, step_back_size = 0, -50, -0.1

# Define the fixed tracking points
tracking_points = [
    (4000, -310), (4200, -310),(4400, -310),(4600, -310),(4800, -310),
    (5200, -310), (5400, -310),(5600, -310),(5800, -310),(6000, -310),
    (5000, -510), (5000, -710),(5000, -110)]

# Track particles for the specified tracking points using the optimized function
all_trajectories = track_particles(tracking_points, tp_start, tp_end, step_size)
all_backward_trajectories = track_particles(tracking_points, tp_back_start, tp_back_end, step_back_size)
    
# Function to load tracked streamlines from CSV
def load_tracked_streamlines(csv_filename):
    df = pd.read_csv(csv_filename)
    streamlines = []
    for streamline_id in df['streamline_id'].unique():
        streamline_data = df[df['streamline_id'] == streamline_id][['x', 'z']].values
        streamlines.append(streamline_data)
    return streamlines

# Load tracked streamlines from CSV
current_working_dir = os.getcwd()
result_folder = os.path.join(current_working_dir, 'results')
csv_filename = os.path.join(result_folder, 'tracked_streamlines.csv')
streamlines = load_tracked_streamlines(csv_filename)

# Plotting setup
fig = go.Figure()

# Color scales for forward and backward trajectories
colorscale_forward = [[0, 'rgba(0, 255, 0, 1)'], [0.5, 'rgba(255, 255, 0, 0.5)'], [1, 'rgba(255, 0, 0, 1)']]
colorscale_backward = [[0, 'rgba(0, 0, 255, 1)'], [0.5, 'rgba(0, 255, 255, 0.5)'], [1, 'rgba(0, 255, 0, 1)']]

# Add traces for forward and backward trajectories
for traj, colorscale in zip([all_trajectories, all_backward_trajectories], [colorscale_forward, colorscale_backward]):
    for path in traj:
        fig.add_trace(go.Scatter(
            x=path[:, 0], 
            y=path[:, 1], 
            mode='lines+markers',
            marker=dict(size=7, color=path[:, 2], colorscale=colorscale, showscale=False),
            name='Forward' if traj is all_trajectories else 'Backward'
        ))

# Plot tracked streamlines
for streamline in streamlines:
    fig.add_trace(go.Scatter(
        x=streamline[:, 0], 
        y=streamline[:, 1], 
        mode='lines', 
        line=dict(width=2, color='black'),
        name='Streamline'
    ))

# Add starting points as black filled circles
start_points_x = [point[0] for point in tracking_points]
start_points_z = [point[1] for point in tracking_points]

fig.add_trace(go.Scatter(
    x=start_points_x,
    y=start_points_z,
    mode='markers',
    marker=dict(size=12, color='black', symbol='circle', line=dict(width=2, color='white')),
    name='Starting Points'
))

# Update layout
fig.update_layout(
    title="Particle Trajectories and Streamlines (Fixed Starting Points)",
    xaxis_title='X',
    yaxis_title='Z',
    margin=dict(r=10, l=10, b=10, t=40),
    xaxis=dict(range=[4500, 5500], dtick=100),
    yaxis=dict(range=[-600, 0], scaleanchor="x", scaleratio=1),
    showlegend=False
)

# Custom color bar for both forward and backward trajectories
fig.update_coloraxes(
    colorscale=[[0, '#0000FF'], [0.5, '#00FF00'], [1, '#FF0000']],
    cmin=-50,
    cmax=50,
    colorbar=dict(
        title='Time (tp)',
        tickvals=[-50, -25, 0, 25, 50],
        ticktext=['-50', '-25', '0', '25', '50']
    )
)

# Show the plot
fig.show()

# Save the plot as an HTML file
html_filename = os.path.join(result_folder, 'basin_scale_particle_trajectories_with_streamlines_taud_0.4_0P_enlarge.html')
fig.write_html(html_filename)

# Optionally, save the trajectory to a CSV file
#csv_filename = os.path.join(result_folder, 'basin_scale_particle_trajectories_with_streamlines_taud_0.4_0P_enlarge.csv')
#np.savetxt(csv_filename, np.vstack([traj.reshape(-1, 3) for traj in all_trajectories + all_backward_trajectories]), delimiter=',', header='x,z,tp', comments='')

print(f"Particle trajectories and streamlines for fixed starting points have been calculated and saved to {html_filename} and {csv_filename}.")

In [ ]:
#Step 6-2:Generate particles trajectories across specified points, accompanied by streamlines under average field. 
#Particles are released from t=0.25P
import numpy as np
import os
import pandas as pd
from numba import jit, prange
import plotly.graph_objects as go

# Parameter settings (ensure these are already defined)
params = {
    'p': 3, 'q': 3, 'HRx': 6, 'HLx': 3, 'D': -2000, 'a': 2/3, 'b': 1/3,
    'LX': 10000, 'LY': 10000, 'TauD': 0.4
}

# Extract parameters
p, q, HRx, HLx, D, a, b, TauD = [params[k] for k in ('p', 'q', 'HRx', 'HLx', 'D', 'a', 'b', 'TauD')]
LX, LY = params['LX'], params['LY']

# Precompute mesh grid and cosine values
mesh_x, mesh_z = 101, 101
x_range = np.linspace(0, LX, mesh_x)
z_range = np.linspace(D, 0, mesh_z)
b1 = np.cos(np.pi * x_range / LX)
b2 = np.cos(p * np.pi * x_range / LX)

# Precompute cosh values for efficiency
c1 = np.cosh(np.pi * D / LX)
c2 = np.cosh(p * np.pi * D / LX)

@jit(nopython=True, parallel=True)
def compute_J(e1, e2):
    """Calculate the J matrix and related variables in 2D (x, z) space."""
    J = np.zeros((mesh_x, mesh_z))
    for i in prange(mesh_x):
        for k in prange(mesh_z):
            hA = HRx - HRx * b1[i] * np.cosh(np.pi * (z_range[k] - D) / LX) / c1
            hB = HLx - HLx * b2[i] * np.cosh(p * np.pi * (z_range[k] - D) / LX) / c2
            g1, g2 = 0.0, 0.0
            for n in range(1, 1000):  
                k1, an = (-1) ** (n - 1), (2 * n - 1) * np.pi / 2
                b0 = np.cos(an * (z_range[k] - D) / D)
                f1 = D * (k1 * an * b0 * (1 / an**2 + (an**2 * e1 + 2 * np.pi * TauD * e2) / (an**4 + 4 * np.pi**2 * TauD**2)))
                g1 += f1
                k2, d1 = (-1) ** n, an**2 + (p * np.pi * D / LX)**2
                f2 = D * (k2 * an * b0 * (1 / d1 + (d1 * e1 + 2 * np.pi * TauD * e2) / (d1**2 + 4 * np.pi**2 * TauD**2)))
                g2 += f2               
            hC = HLx * g1 / D + HLx * b2[i] * g2 / D
            J[i, k] = hA + a * hB + b * hC
    return J

def calculate_fields(tp):
    e1 = np.cos(2 * np.pi * tp)
    e2 = np.sin(2 * np.pi * tp)
    J = compute_J(e1, e2)
    dx, dz = x_range[1] - x_range[0], z_range[1] - z_range[0]
    grad_x, grad_z = np.gradient(J, dx, dz)
    vx, vz = -100000 * grad_x, -100000 * grad_z
    H = J  # Hydraulic head (H) is equivalent to J
    return H, vx, vz  # Removed UU from the return statement

# Function to get velocity components at a specific point (x, z) without recalculating the entire field each time
def get_velocity_components(vx, vz, x, z):
    """Retrieve the velocity components at a specific point (x, z)."""
    ix = np.abs(x_range - x).argmin()
    iz = np.abs(z_range - z).argmin()
    return vx[ix, iz], vz[ix, iz]

def track_particles(tracking_points, tp_start, tp_end, step_size, x_range=(0, 10000), z_range=(-2000, -40)):
    positions = []
    # Precompute the velocity fields for all time points once and store them in a dictionary
    if step_size > 0:
        time_points = np.round(np.arange(tp_start, tp_end + step_size, step_size), decimals=2)
    else:
        time_points = np.round(np.arange(tp_start, tp_end - step_size, step_size), decimals=2)  # For backward tracking

    velocity_fields = {tp: calculate_fields(tp)[1:] for tp in time_points}
    
    for x_start, z_start in tracking_points:
        x, z = x_start, z_start
        traj = [(x, z, tp_start)]
        
        if step_size > 0:
            iter_time_points = time_points[time_points > tp_start]
        else:
            iter_time_points = time_points[time_points < tp_start]  # For backward tracking
        
        for tp in iter_time_points:
            try:
                vx, vz = velocity_fields[tp]
            except KeyError as e:
                print(f"KeyError encountered: {e}. Available keys: {list(velocity_fields.keys())}")
                raise
            vx_at_point, vz_at_point = get_velocity_components(vx, vz, x, z)
            x_new, z_new = x + vx_at_point * step_size, z + vz_at_point * step_size
            
            if not (x_range[0] <= x_new <= x_range[1] and z_range[0] <= z_new <= z_range[1]):
                break
            
            traj.append((x_new, z_new, tp))
            x, z = x_new, z_new
        
        positions.append(np.array(traj))
    
    return positions

# Set initial particle position and time parameters
tp_start, tp_end, step_size = 0.25, 50, 0.1
tp_back_start, tp_back_end, step_back_size = 0.25, -50, -0.1

# Define the fixed tracking points
tracking_points = [
    (4000, -310), (4200, -310),(4400, -310),(4600, -310),(4800, -310),
    (5200, -310), (5400, -310),(5600, -310),(5800, -310),(6000, -310),
    (5000, -510), (5000, -710),(5000, -110)]

# Track particles for the specified tracking points using the optimized function
all_trajectories = track_particles(tracking_points, tp_start, tp_end, step_size)
all_backward_trajectories = track_particles(tracking_points, tp_back_start, tp_back_end, step_back_size)
    
# Function to load tracked streamlines from CSV
def load_tracked_streamlines(csv_filename):
    df = pd.read_csv(csv_filename)
    streamlines = []
    for streamline_id in df['streamline_id'].unique():
        streamline_data = df[df['streamline_id'] == streamline_id][['x', 'z']].values
        streamlines.append(streamline_data)
    return streamlines

# Load tracked streamlines from CSV
current_working_dir = os.getcwd()
result_folder = os.path.join(current_working_dir, 'results')
csv_filename = os.path.join(result_folder, 'tracked_streamlines.csv')
streamlines = load_tracked_streamlines(csv_filename)

# Plotting setup
fig = go.Figure()

# Color scales for forward and backward trajectories
colorscale_forward = [[0, 'rgba(0, 255, 0, 1)'], [0.5, 'rgba(255, 255, 0, 0.5)'], [1, 'rgba(255, 0, 0, 1)']]
colorscale_backward = [[0, 'rgba(0, 0, 255, 1)'], [0.5, 'rgba(0, 255, 255, 0.5)'], [1, 'rgba(0, 255, 0, 1)']]

# Add traces for forward and backward trajectories
for traj, colorscale in zip([all_trajectories, all_backward_trajectories], [colorscale_forward, colorscale_backward]):
    for path in traj:
        fig.add_trace(go.Scatter(
            x=path[:, 0], 
            y=path[:, 1], 
            mode='lines+markers',
            marker=dict(size=5, color=path[:, 2], colorscale=colorscale, showscale=False),
            name='Forward' if traj is all_trajectories else 'Backward'
        ))

# Plot tracked streamlines
for streamline in streamlines:
    fig.add_trace(go.Scatter(
        x=streamline[:, 0], 
        y=streamline[:, 1], 
        mode='lines', 
        line=dict(width=2, color='black'),
        name='Streamline'
    ))

# Add starting points as black filled circles
start_points_x = [point[0] for point in tracking_points]
start_points_z = [point[1] for point in tracking_points]

fig.add_trace(go.Scatter(
    x=start_points_x,
    y=start_points_z,
    mode='markers',
    marker=dict(size=8, color='black', symbol='circle', line=dict(width=2, color='white')),
    name='Starting Points'
))

# Update layout
fig.update_layout(
    title="Particle Trajectories and Streamlines (Fixed Starting Points)",
    xaxis_title='X',
    yaxis_title='Z',
    margin=dict(r=10, l=10, b=10, t=40),
    xaxis=dict(range=[0, 10000]),
    yaxis=dict(range=[-2000, -40], scaleanchor="x", scaleratio=1),
    showlegend=False
)

# Custom color bar for both forward and backward trajectories
fig.update_coloraxes(
    colorscale=[[0, '#0000FF'], [0.5, '#00FF00'], [1, '#FF0000']],
    cmin=-50,
    cmax=50,
    colorbar=dict(
        title='Time (tp)',
        tickvals=[-50, -25, 0, 25, 50],
        ticktext=['-50', '-25', '0', '25', '50']
    )
)

# Show the plot
fig.show()

# Save the plot as an HTML file
html_filename = os.path.join(result_folder, 'basin_scale_particle_trajectories_with_streamlines_taud_0.4_0.25P.html')
fig.write_html(html_filename)

# Optionally, save the trajectory to a CSV file
#csv_filename = os.path.join(result_folder, 'basin_scale_particle_trajectories_with_streamlines_taud_0.4_0.25P.csv')
#np.savetxt(csv_filename, np.vstack([traj.reshape(-1, 3) for traj in all_trajectories + all_backward_trajectories]), delimiter=',', header='x,z,tp', comments='')

print(f"Particle trajectories and streamlines for fixed starting points have been calculated and saved to {html_filename} and {csv_filename}.")

In [ ]:
#Step 6-3:Generate particles trajectories across specified points, accompanied by streamlines under average field. 
#Particles are released from t=0.5P
import numpy as np
import os
import pandas as pd
from numba import jit, prange
import plotly.graph_objects as go

# Parameter settings (ensure these are already defined)
params = {
    'p': 3, 'q': 3, 'HRx': 6, 'HLx': 3, 'D': -2000, 'a': 2/3, 'b': 1/3,
    'LX': 10000, 'LY': 10000, 'TauD': 0.4
}

# Extract parameters
p, q, HRx, HLx, D, a, b, TauD = [params[k] for k in ('p', 'q', 'HRx', 'HLx', 'D', 'a', 'b', 'TauD')]
LX, LY = params['LX'], params['LY']

# Precompute mesh grid and cosine values
mesh_x, mesh_z = 101, 101
x_range = np.linspace(0, LX, mesh_x)
z_range = np.linspace(D, 0, mesh_z)
b1 = np.cos(np.pi * x_range / LX)
b2 = np.cos(p * np.pi * x_range / LX)

# Precompute cosh values for efficiency
c1 = np.cosh(np.pi * D / LX)
c2 = np.cosh(p * np.pi * D / LX)

@jit(nopython=True, parallel=True)
def compute_J(e1, e2):
    """Calculate the J matrix and related variables in 2D (x, z) space."""
    J = np.zeros((mesh_x, mesh_z))
    for i in prange(mesh_x):
        for k in prange(mesh_z):
            hA = HRx - HRx * b1[i] * np.cosh(np.pi * (z_range[k] - D) / LX) / c1
            hB = HLx - HLx * b2[i] * np.cosh(p * np.pi * (z_range[k] - D) / LX) / c2
            g1, g2 = 0.0, 0.0
            for n in range(1, 1000):  
                k1, an = (-1) ** (n - 1), (2 * n - 1) * np.pi / 2
                b0 = np.cos(an * (z_range[k] - D) / D)
                f1 = D * (k1 * an * b0 * (1 / an**2 + (an**2 * e1 + 2 * np.pi * TauD * e2) / (an**4 + 4 * np.pi**2 * TauD**2)))
                g1 += f1
                k2, d1 = (-1) ** n, an**2 + (p * np.pi * D / LX)**2
                f2 = D * (k2 * an * b0 * (1 / d1 + (d1 * e1 + 2 * np.pi * TauD * e2) / (d1**2 + 4 * np.pi**2 * TauD**2)))
                g2 += f2               
            hC = HLx * g1 / D + HLx * b2[i] * g2 / D
            J[i, k] = hA + a * hB + b * hC
    return J

def calculate_fields(tp):
    e1 = np.cos(2 * np.pi * tp)
    e2 = np.sin(2 * np.pi * tp)
    J = compute_J(e1, e2)
    dx, dz = x_range[1] - x_range[0], z_range[1] - z_range[0]
    grad_x, grad_z = np.gradient(J, dx, dz)
    vx, vz = -100000 * grad_x, -100000 * grad_z
    H = J  # Hydraulic head (H) is equivalent to J
    return H, vx, vz  # Removed UU from the return statement

# Function to get velocity components at a specific point (x, z) without recalculating the entire field each time
def get_velocity_components(vx, vz, x, z):
    """Retrieve the velocity components at a specific point (x, z)."""
    ix = np.abs(x_range - x).argmin()
    iz = np.abs(z_range - z).argmin()
    return vx[ix, iz], vz[ix, iz]

def track_particles(tracking_points, tp_start, tp_end, step_size, x_range=(0, 10000), z_range=(-2000, -40)):
    positions = []
    # Precompute the velocity fields for all time points once and store them in a dictionary
    if step_size > 0:
        time_points = np.round(np.arange(tp_start, tp_end + step_size, step_size), decimals=2)
    else:
        time_points = np.round(np.arange(tp_start, tp_end - step_size, step_size), decimals=2)  # For backward tracking

    velocity_fields = {tp: calculate_fields(tp)[1:] for tp in time_points}
    
    for x_start, z_start in tracking_points:
        x, z = x_start, z_start
        traj = [(x, z, tp_start)]
        
        if step_size > 0:
            iter_time_points = time_points[time_points > tp_start]
        else:
            iter_time_points = time_points[time_points < tp_start]  # For backward tracking
        
        for tp in iter_time_points:
            try:
                vx, vz = velocity_fields[tp]
            except KeyError as e:
                print(f"KeyError encountered: {e}. Available keys: {list(velocity_fields.keys())}")
                raise
            vx_at_point, vz_at_point = get_velocity_components(vx, vz, x, z)
            x_new, z_new = x + vx_at_point * step_size, z + vz_at_point * step_size
            
            if not (x_range[0] <= x_new <= x_range[1] and z_range[0] <= z_new <= z_range[1]):
                break
            
            traj.append((x_new, z_new, tp))
            x, z = x_new, z_new
        
        positions.append(np.array(traj))
    
    return positions

# Set initial particle position and time parameters
tp_start, tp_end, step_size = 0.5, 50, 0.1
tp_back_start, tp_back_end, step_back_size = 0.5, -50, -0.1

# Define the fixed tracking points
tracking_points = [
    (4000, -310), (4200, -310),(4400, -310),(4600, -310),(4800, -310),
    (5200, -310), (5400, -310),(5600, -310),(5800, -310),(6000, -310),
    (5000, -510), (5000, -710),(5000, -110)]

# Track particles for the specified tracking points using the optimized function
all_trajectories = track_particles(tracking_points, tp_start, tp_end, step_size)
all_backward_trajectories = track_particles(tracking_points, tp_back_start, tp_back_end, step_back_size)
    
# Function to load tracked streamlines from CSV
def load_tracked_streamlines(csv_filename):
    df = pd.read_csv(csv_filename)
    streamlines = []
    for streamline_id in df['streamline_id'].unique():
        streamline_data = df[df['streamline_id'] == streamline_id][['x', 'z']].values
        streamlines.append(streamline_data)
    return streamlines

# Load tracked streamlines from CSV
current_working_dir = os.getcwd()
result_folder = os.path.join(current_working_dir, 'results')
csv_filename = os.path.join(result_folder, 'tracked_streamlines.csv')
streamlines = load_tracked_streamlines(csv_filename)

# Plotting setup
fig = go.Figure()

# Color scales for forward and backward trajectories
colorscale_forward = [[0, 'rgba(0, 255, 0, 1)'], [0.5, 'rgba(255, 255, 0, 0.5)'], [1, 'rgba(255, 0, 0, 1)']]
colorscale_backward = [[0, 'rgba(0, 0, 255, 1)'], [0.5, 'rgba(0, 255, 255, 0.5)'], [1, 'rgba(0, 255, 0, 1)']]

# Add traces for forward and backward trajectories
for traj, colorscale in zip([all_trajectories, all_backward_trajectories], [colorscale_forward, colorscale_backward]):
    for path in traj:
        fig.add_trace(go.Scatter(
            x=path[:, 0], 
            y=path[:, 1], 
            mode='lines+markers',
            marker=dict(size=5, color=path[:, 2], colorscale=colorscale, showscale=False),
            name='Forward' if traj is all_trajectories else 'Backward'
        ))

# Plot tracked streamlines
for streamline in streamlines:
    fig.add_trace(go.Scatter(
        x=streamline[:, 0], 
        y=streamline[:, 1], 
        mode='lines', 
        line=dict(width=2, color='black'),
        name='Streamline'
    ))

# Add starting points as black filled circles
start_points_x = [point[0] for point in tracking_points]
start_points_z = [point[1] for point in tracking_points]

fig.add_trace(go.Scatter(
    x=start_points_x,
    y=start_points_z,
    mode='markers',
    marker=dict(size=8, color='black', symbol='circle', line=dict(width=2, color='white')),
    name='Starting Points'
))

# Update layout
fig.update_layout(
    title="Particle Trajectories and Streamlines (Fixed Starting Points)",
    xaxis_title='X',
    yaxis_title='Z',
    margin=dict(r=10, l=10, b=10, t=40),
    xaxis=dict(range=[0, 10000]),
    yaxis=dict(range=[-2000, -40], scaleanchor="x", scaleratio=1),
    showlegend=False
)

# Custom color bar for both forward and backward trajectories
fig.update_coloraxes(
    colorscale=[[0, '#0000FF'], [0.5, '#00FF00'], [1, '#FF0000']],
    cmin=-50,
    cmax=50,
    colorbar=dict(
        title='Time (tp)',
        tickvals=[-50, -25, 0, 25, 50],
        ticktext=['-50', '-25', '0', '25', '50']
    )
)

# Show the plot
fig.show()

# Save the plot as an HTML file
html_filename = os.path.join(result_folder, 'basin_scale_particle_trajectories_with_streamlines_taud_0.4_0.5P.html')
fig.write_html(html_filename)

# Optionally, save the trajectory to a CSV file
#csv_filename = os.path.join(result_folder, 'basin_scale_particle_trajectories_with_streamlines_taud_0.4_0.5P.csv')
#np.savetxt(csv_filename, np.vstack([traj.reshape(-1, 3) for traj in all_trajectories + all_backward_trajectories]), delimiter=',', header='x,z,tp', comments='')

print(f"Particle trajectories and streamlines for fixed starting points have been calculated and saved to {html_filename} and {csv_filename}.")

In [ ]:
#Step 6-4:Generate basin-scale particles trajectories across specified points, accompanied by streamlines under average field. 
#Particles are released from t=0.75P
import numpy as np
import os
import pandas as pd
from numba import jit, prange
import plotly.graph_objects as go

# Parameter settings (ensure these are already defined)
params = {
    'p': 3, 'q': 3, 'HRx': 6, 'HLx': 3, 'D': -2000, 'a': 2/3, 'b': 1/3,
    'LX': 10000, 'LY': 10000, 'TauD': 0.4
}

# Extract parameters
p, q, HRx, HLx, D, a, b, TauD = [params[k] for k in ('p', 'q', 'HRx', 'HLx', 'D', 'a', 'b', 'TauD')]
LX, LY = params['LX'], params['LY']

# Precompute mesh grid and cosine values
mesh_x, mesh_z = 101, 101
x_range = np.linspace(0, LX, mesh_x)
z_range = np.linspace(D, 0, mesh_z)
b1 = np.cos(np.pi * x_range / LX)
b2 = np.cos(p * np.pi * x_range / LX)

# Precompute cosh values for efficiency
c1 = np.cosh(np.pi * D / LX)
c2 = np.cosh(p * np.pi * D / LX)

@jit(nopython=True, parallel=True)
def compute_J(e1, e2):
    """Calculate the J matrix and related variables in 2D (x, z) space."""
    J = np.zeros((mesh_x, mesh_z))
    for i in prange(mesh_x):
        for k in prange(mesh_z):
            hA = HRx - HRx * b1[i] * np.cosh(np.pi * (z_range[k] - D) / LX) / c1
            hB = HLx - HLx * b2[i] * np.cosh(p * np.pi * (z_range[k] - D) / LX) / c2
            g1, g2 = 0.0, 0.0
            for n in range(1, 1000):  
                k1, an = (-1) ** (n - 1), (2 * n - 1) * np.pi / 2
                b0 = np.cos(an * (z_range[k] - D) / D)
                f1 = D * (k1 * an * b0 * (1 / an**2 + (an**2 * e1 + 2 * np.pi * TauD * e2) / (an**4 + 4 * np.pi**2 * TauD**2)))
                g1 += f1
                k2, d1 = (-1) ** n, an**2 + (p * np.pi * D / LX)**2
                f2 = D * (k2 * an * b0 * (1 / d1 + (d1 * e1 + 2 * np.pi * TauD * e2) / (d1**2 + 4 * np.pi**2 * TauD**2)))
                g2 += f2               
            hC = HLx * g1 / D + HLx * b2[i] * g2 / D
            J[i, k] = hA + a * hB + b * hC
    return J

def calculate_fields(tp):
    e1 = np.cos(2 * np.pi * tp)
    e2 = np.sin(2 * np.pi * tp)
    J = compute_J(e1, e2)
    dx, dz = x_range[1] - x_range[0], z_range[1] - z_range[0]
    grad_x, grad_z = np.gradient(J, dx, dz)
    vx, vz = -100000 * grad_x, -100000 * grad_z
    H = J  # Hydraulic head (H) is equivalent to J
    return H, vx, vz  # Removed UU from the return statement

# Function to get velocity components at a specific point (x, z) without recalculating the entire field each time
def get_velocity_components(vx, vz, x, z):
    """Retrieve the velocity components at a specific point (x, z)."""
    ix = np.abs(x_range - x).argmin()
    iz = np.abs(z_range - z).argmin()
    return vx[ix, iz], vz[ix, iz]

def track_particles(tracking_points, tp_start, tp_end, step_size, x_range=(0, 10000), z_range=(-2000, -40)):
    positions = []
    # Precompute the velocity fields for all time points once and store them in a dictionary
    if step_size > 0:
        time_points = np.round(np.arange(tp_start, tp_end + step_size, step_size), decimals=2)
    else:
        time_points = np.round(np.arange(tp_start, tp_end - step_size, step_size), decimals=2)  # For backward tracking

    velocity_fields = {tp: calculate_fields(tp)[1:] for tp in time_points}
    
    for x_start, z_start in tracking_points:
        x, z = x_start, z_start
        traj = [(x, z, tp_start)]
        
        if step_size > 0:
            iter_time_points = time_points[time_points > tp_start]
        else:
            iter_time_points = time_points[time_points < tp_start]  # For backward tracking
        
        for tp in iter_time_points:
            try:
                vx, vz = velocity_fields[tp]
            except KeyError as e:
                print(f"KeyError encountered: {e}. Available keys: {list(velocity_fields.keys())}")
                raise
            vx_at_point, vz_at_point = get_velocity_components(vx, vz, x, z)
            x_new, z_new = x + vx_at_point * step_size, z + vz_at_point * step_size
            
            if not (x_range[0] <= x_new <= x_range[1] and z_range[0] <= z_new <= z_range[1]):
                break
            
            traj.append((x_new, z_new, tp))
            x, z = x_new, z_new
        
        positions.append(np.array(traj))
    
    return positions

# Set initial particle position and time parameters
tp_start, tp_end, step_size = 0.75, 50, 0.1
tp_back_start, tp_back_end, step_back_size = 0.75, -50, -0.1

# Define the fixed tracking points
tracking_points = [
    (4000, -310), (4200, -310),(4400, -310),(4600, -310),(4800, -310),
    (5200, -310), (5400, -310),(5600, -310),(5800, -310),(6000, -310),
    (5000, -510), (5000, -710),(5000, -110)]

# Track particles for the specified tracking points using the optimized function
all_trajectories = track_particles(tracking_points, tp_start, tp_end, step_size)
all_backward_trajectories = track_particles(tracking_points, tp_back_start, tp_back_end, step_back_size)
    
# Function to load tracked streamlines from CSV
def load_tracked_streamlines(csv_filename):
    df = pd.read_csv(csv_filename)
    streamlines = []
    for streamline_id in df['streamline_id'].unique():
        streamline_data = df[df['streamline_id'] == streamline_id][['x', 'z']].values
        streamlines.append(streamline_data)
    return streamlines

# Load tracked streamlines from CSV
current_working_dir = os.getcwd()
result_folder = os.path.join(current_working_dir, 'results')
csv_filename = os.path.join(result_folder, 'tracked_streamlines.csv')
streamlines = load_tracked_streamlines(csv_filename)

# Plotting setup
fig = go.Figure()

# Color scales for forward and backward trajectories
colorscale_forward = [[0, 'rgba(0, 255, 0, 1)'], [0.5, 'rgba(255, 255, 0, 0.5)'], [1, 'rgba(255, 0, 0, 1)']]
colorscale_backward = [[0, 'rgba(0, 0, 255, 1)'], [0.5, 'rgba(0, 255, 255, 0.5)'], [1, 'rgba(0, 255, 0, 1)']]

# Add traces for forward and backward trajectories
for traj, colorscale in zip([all_trajectories, all_backward_trajectories], [colorscale_forward, colorscale_backward]):
    for path in traj:
        fig.add_trace(go.Scatter(
            x=path[:, 0], 
            y=path[:, 1], 
            mode='lines+markers',
            marker=dict(size=5, color=path[:, 2], colorscale=colorscale, showscale=False),
            name='Forward' if traj is all_trajectories else 'Backward'
        ))

# Plot tracked streamlines
for streamline in streamlines:
    fig.add_trace(go.Scatter(
        x=streamline[:, 0], 
        y=streamline[:, 1], 
        mode='lines', 
        line=dict(width=2, color='black'),
        name='Streamline'
    ))

# Add starting points as black filled circles
start_points_x = [point[0] for point in tracking_points]
start_points_z = [point[1] for point in tracking_points]

fig.add_trace(go.Scatter(
    x=start_points_x,
    y=start_points_z,
    mode='markers',
    marker=dict(size=8, color='black', symbol='circle', line=dict(width=2, color='white')),
    name='Starting Points'
))

# Update layout
fig.update_layout(
    title="Particle Trajectories and Streamlines (Fixed Starting Points)",
    xaxis_title='X',
    yaxis_title='Z',
    margin=dict(r=10, l=10, b=10, t=40),
    xaxis=dict(range=[0, 10000]),
    yaxis=dict(range=[-2000, -40], scaleanchor="x", scaleratio=1),
    showlegend=False
)

# Custom color bar for both forward and backward trajectories
fig.update_coloraxes(
    colorscale=[[0, '#0000FF'], [0.5, '#00FF00'], [1, '#FF0000']],
    cmin=-50,
    cmax=50,
    colorbar=dict(
        title='Time (tp)',
        tickvals=[-50, -25, 0, 25, 50],
        ticktext=['-50', '-25', '0', '25', '50']
    )
)

# Show the plot
fig.show()

# Save the plot as an HTML file
html_filename = os.path.join(result_folder, 'basin_scale_particle_trajectories_with_streamlines_taud_0.4_0.75P.html')
fig.write_html(html_filename)

# Optionally, save the trajectory to a CSV file
#csv_filename = os.path.join(result_folder, 'basin_scale_particle_trajectories_with_streamlines_taud_0.4_0.75P.csv')
#np.savetxt(csv_filename, np.vstack([traj.reshape(-1, 3) for traj in all_trajectories + all_backward_trajectories]), delimiter=',', header='x,z,tp', comments='')

print(f"Particle trajectories and streamlines for fixed starting points have been calculated and saved to {html_filename} and {csv_filename}.")

In [ ]:
#Step 7-1: Generate local-scale trajectories across specified points.5 points with stagnant zone.
import numpy as np
import os
import pandas as pd
from pathlib import Path
import h5py
from numba import jit, prange
import plotly.graph_objects as go

# Parameter settings (ensure these are already defined)
params = {
    'p': 3, 'q': 3, 'HRx': 6, 'HLx': 3, 'D': -2000, 'a': 2/3, 'b': 1/3,
    'LX': 10000, 'LY': 10000, 'TauD': 0.4
}

# Extract parameters
p, q, HRx, HLx, D, a, b, TauD = params['p'], params['q'], params['HRx'], params['HLx'], params['D'], params['a'], params['b'], params['TauD']
LX, LY = params['LX'], params['LY']

# Define the mesh grid (only for x and z directions)
mesh_x, mesh_z = 101, 101
x_range = np.linspace(0, LX, mesh_x)
z_range = np.linspace(D, 0, mesh_z)

# Precompute cosine values
b1 = np.cos(np.pi * x_range / LX)
b2 = np.cos(p * np.pi * x_range / LX)

@jit(nopython=True, parallel=True)
def compute_J(mesh_x, mesh_z, x_range, z_range, HRx, HLx, LX, D, a, b, TauD, b1, b2, c1, c2, e1, e2):
    J = np.zeros((mesh_x, mesh_z))
    for i in prange(mesh_x):
        for k in prange(mesh_z):
            hA = HRx - HRx * b1[i] * np.cosh(np.pi * (z_range[k] - D) / LX) / c1
            hB = HLx - HLx * b2[i] * np.cosh(p * np.pi * (z_range[k] - D) / LX) / c2
            g1, g2 = 0.0, 0.0
            for n in range(1, 1000):  
                k1, an = (-1) ** (n - 1), (2 * n - 1) * np.pi / 2
                b0 = np.cos(an * (z_range[k] - D) / D)
                f1 = D * (k1 * an * b0 * (1 / an**2 + (an**2 * e1 + 2 * np.pi * TauD * e2) / (an**4 + 4 * np.pi**2 * TauD**2)))
                g1 += f1
                k2, d1 = (-1) ** n, an**2 + (p * np.pi * D / LX)**2
                f2 = D * (k2 * an * b0 * (1 / d1 + (d1 * e1 + 2 * np.pi * TauD * e2) / (d1**2 + 4 * np.pi**2 * TauD**2)))
                g2 += f2               
            hC = HLx * g1 / D + HLx * b2[i] * g2 / D
            J[i, k] = hA + a * hB + b * hC
    return J

def calculate_fields(tp):
    e1 = np.cos(2 * np.pi * tp)
    e2 = np.sin(2 * np.pi * tp)
    c1 = np.cosh(np.pi * D / LX)
    c2 = np.cosh(p * np.pi * D / LX)
    J = compute_J(mesh_x, mesh_z, x_range, z_range, HRx, HLx, LX, D, a, b, TauD, b1, b2, c1, c2, e1, e2)
    dx, dz = x_range[1] - x_range[0], z_range[1] - z_range[0]
    grad_x, grad_z = np.gradient(J, dx, dz)
    vx, vz = -100000*grad_x, -100000*grad_z
    UU = np.sqrt(vx**2 + vz**2)
    return J, vx, vz, UU

def track_particle(x_start, z_start, tp_start, tp_end, step_size):
    x, z = x_start, z_start
    positions = [(x, z, tp_start)]
    for tp in np.arange(tp_start + step_size, tp_end, step_size):
        vx, vz = get_velocity_at_point(x, z, tp)
        x_new = x + vx * step_size
        z_new = z + vz * step_size
        if z_new >= 0:
            break
        positions.append((x_new, z_new, tp))
        x, z = x_new, z_new
    return np.array(positions)

# Function to calculate the velocity at a given point (x, z) for a given tp
def get_velocity_at_point(x, z, tp):
    """Calculate the velocity components at a specific point (x, z) at time tp."""
    # Get the velocity field at the current time point
    H, vx, vz, UU = calculate_fields(tp)
    
    # Find the closest indices for (x, z) in the grid
    ix = np.abs(x_range - x).argmin()
    iz = np.abs(z_range - z).argmin()
    
    # Return the velocity components at the closest point
    return vx[ix, iz], vz[ix, iz]

def main():
    tp_start, tp_end, step_size = 0, 10.1, 0.1
    x_start_values = [5000,5100,5300]
    z_start_values = [-310]
    all_trajectories = []
    for x_start in x_start_values:
        for z_start in z_start_values:
            trajectory = track_particle(x_start, z_start, tp_start, tp_end, step_size)
            all_trajectories.append(trajectory)

    current_working_dir = os.getcwd()
    result_folder = os.path.join(current_working_dir, 'results')
    os.makedirs(result_folder, exist_ok=True)
    
    # Save trajectories to CSV
    x_all = np.concatenate([traj[:, 0] for traj in all_trajectories])
    z_all = np.concatenate([traj[:, 1] for traj in all_trajectories])
    tp_all = np.concatenate([traj[:, 2] for traj in all_trajectories])
    trajectory_filename_csv = os.path.join(result_folder, 'trajectories_aroundsp.csv')
    np.savetxt(trajectory_filename_csv, np.column_stack([x_all, z_all, tp_all]), delimiter=',', header='x,z,tp', comments='')
    print(f"Saved to {trajectory_filename_csv}.")

    # Save trajectories to .dat file in Tecplot format
    trajectory_filename_dat = os.path.join(result_folder, 'trajectories_aroundsp.dat')
    with open(trajectory_filename_dat, 'w') as f:
        f.write('VARIABLES = "X", "Z", "TP"\n')
        f.write(f"ZONE T='matrix_particle_trajectories_2d_tp0'\n")
        f.write(f'I={len(x_all)}, J=1, K=1, F=POINT\n')
        np.savetxt(f, np.column_stack([x_all, z_all, tp_all]), fmt='%f', delimiter='\t')
    print(f"Particle trajectories have also been saved to {trajectory_filename_dat}.")

    # Load stagnation points from CSV
    stagnation_points_path = os.path.join(result_folder, 'all_stagnation_points.csv')
    df_stagnation = pd.read_csv(stagnation_points_path)

    fig = go.Figure()

    for traj in all_trajectories:
        x_traj, z_traj, tp_traj = traj[:, 0], traj[:, 1], traj[:, 2]
        
        # Add particle trajectories
        fig.add_trace(go.Scatter(x=x_traj, y=z_traj, mode='lines+markers',
                                 marker=dict(size=5, color=tp_traj, colorscale=[[0, 'rgba(0, 255, 0, 1)'], [0.5, 'rgba(255, 255, 0, 0.5)'], [1, 'rgba(255, 0, 0, 1)']],
                                 colorbar=dict(title='Time (tp)'))))
        
        # Add starting points as black filled circles
        fig.add_trace(go.Scatter(x=[x_traj[0]], y=[z_traj[0]], mode='markers',
                                 marker=dict(size=8, color='black', symbol='circle', line=dict(width=2, color='white')),
                                 name='Starting Points'))

    # Add stagnation points as black circles
    x_stag, z_stag = df_stagnation['x'], df_stagnation['z']
    fig.add_trace(go.Scatter(x=x_stag, y=z_stag, mode='markers',
                             marker=dict(size=10, color='black', symbol='circle-open'),
                             name='Stagnation Points'))
    
    # Draw stagnation points closed loop
#    fig.add_trace(go.Scatter(x=np.append(x_stag, x_stag[0]), y=np.append(z_stag, z_stag[0]), mode='lines',
#                             line=dict(color='black', width=2, dash='dash'),
#                             name='Stagnation Closed Loop'))
    fig.add_trace(go.Scatter(x=x_stag, y=z_stag, mode='lines',
                         line=dict(color='black', width=2, dash='dash'),
                         name='Stagnation Points Line'))

    fig.update_layout(title="Particle Trajectories with Stagnation Points",
                      xaxis_title='X',
                      yaxis_title='Z',
                      margin=dict(r=10, l=10, b=10, t=40),
                      xaxis=dict(
                          range=[4500, 5500],
                          dtick=100,  
                          scaleanchor="y",  
                          scaleratio=1      
                      ),
                      yaxis=dict(range=[-800, 0]),
                      showlegend=False)

    html_filename = os.path.join(result_folder, 'trajectories_aroundsp.html')
    fig.write_html(html_filename)
    print(f"Saved to {html_filename}.")
    fig.show()

if __name__ == "__main__":
    main()


In [ ]:
#Step 7-2: Generate local-scale trajectories across specified points.without stagnant zone.point at x=5300
import numpy as np
import os
import pandas as pd
from pathlib import Path
import h5py
from numba import jit, prange
import plotly.graph_objects as go

# Parameter settings (ensure these are already defined)
params = {
    'p': 3, 'q': 3, 'HRx': 6, 'HLx': 3, 'D': -2000, 'a': 2/3, 'b': 1/3,
    'LX': 10000, 'LY': 10000, 'TauD': 0.4
}

# Extract parameters
p, q, HRx, HLx, D, a, b, TauD = params['p'], params['q'], params['HRx'], params['HLx'], params['D'], params['a'], params['b'], params['TauD']
LX, LY = params['LX'], params['LY']

# Define the mesh grid (only for x and z directions)
mesh_x, mesh_z = 101, 101
x_range = np.linspace(0, LX, mesh_x)
z_range = np.linspace(D, 0, mesh_z)

# Precompute cosine values
b1 = np.cos(np.pi * x_range / LX)
b2 = np.cos(p * np.pi * x_range / LX)

@jit(nopython=True, parallel=True)
def compute_J(mesh_x, mesh_z, x_range, z_range, HRx, HLx, LX, D, a, b, TauD, b1, b2, c1, c2, e1, e2):
    J = np.zeros((mesh_x, mesh_z))
    for i in prange(mesh_x):
        for k in prange(mesh_z):
            hA = HRx - HRx * b1[i] * np.cosh(np.pi * (z_range[k] - D) / LX) / c1
            hB = HLx - HLx * b2[i] * np.cosh(p * np.pi * (z_range[k] - D) / LX) / c2
            g1, g2 = 0.0, 0.0
            for n in range(1, 1000):  
                k1, an = (-1) ** (n - 1), (2 * n - 1) * np.pi / 2
                b0 = np.cos(an * (z_range[k] - D) / D)
                f1 = D * (k1 * an * b0 * (1 / an**2 + (an**2 * e1 + 2 * np.pi * TauD * e2) / (an**4 + 4 * np.pi**2 * TauD**2)))
                g1 += f1
                k2, d1 = (-1) ** n, an**2 + (p * np.pi * D / LX)**2
                f2 = D * (k2 * an * b0 * (1 / d1 + (d1 * e1 + 2 * np.pi * TauD * e2) / (d1**2 + 4 * np.pi**2 * TauD**2)))
                g2 += f2               
            hC = HLx * g1 / D + HLx * b2[i] * g2 / D
            J[i, k] = hA + a * hB + b * hC
    return J

def calculate_fields(tp):
    e1 = np.cos(2 * np.pi * tp)
    e2 = np.sin(2 * np.pi * tp)
    c1 = np.cosh(np.pi * D / LX)
    c2 = np.cosh(p * np.pi * D / LX)
    J = compute_J(mesh_x, mesh_z, x_range, z_range, HRx, HLx, LX, D, a, b, TauD, b1, b2, c1, c2, e1, e2)
    dx, dz = x_range[1] - x_range[0], z_range[1] - z_range[0]
    grad_x, grad_z = np.gradient(J, dx, dz)
    vx, vz = -100000*grad_x, -100000*grad_z
    UU = np.sqrt(vx**2 + vz**2)
    return J, vx, vz, UU

# Function to calculate the velocity at a given point (x, z) for a given tp
def get_velocity_at_point(x, z, tp):
    """Calculate the velocity components at a specific point (x, z) at time tp."""
    # Get the velocity field at the current time point
    H, vx, vz, UU = calculate_fields(tp)
    
    # Find the closest indices for (x, z) in the grid
    ix = np.abs(x_range - x).argmin()
    iz = np.abs(z_range - z).argmin()
    
    # Return the velocity components at the closest point
    return vx[ix, iz], vz[ix, iz]

def track_particle(x_start, z_start, tp_start, tp_end, step_size):
    x, z = x_start, z_start
    positions = [(x, z, tp_start)]
    for tp in np.arange(tp_start + step_size, tp_end, step_size):
        vx, vz = get_velocity_at_point(x, z, tp)
        x_new = x + vx * step_size
        z_new = z + vz * step_size
        if z_new >= 0:
            break
        positions.append((x_new, z_new, tp))
        x, z = x_new, z_new
    return np.array(positions)

def main():
    tp_start, tp_end, step_size = 0, 10.1, 0.1
    x_start_values = [5300]
    z_start_values = [-310]
    all_trajectories = []
    for x_start in x_start_values:
        for z_start in z_start_values:
            trajectory = track_particle(x_start, z_start, tp_start, tp_end, step_size)
            all_trajectories.append(trajectory)

    current_working_dir = os.getcwd()
    result_folder = os.path.join(current_working_dir, 'results')
    os.makedirs(result_folder, exist_ok=True)
    
    # Save trajectories to CSV
    x_all = np.concatenate([traj[:, 0] for traj in all_trajectories])
    z_all = np.concatenate([traj[:, 1] for traj in all_trajectories])
    tp_all = np.concatenate([traj[:, 2] for traj in all_trajectories])
    trajectory_filename_csv = os.path.join(result_folder, 'trajectories_x5300.csv')
    np.savetxt(trajectory_filename_csv, np.column_stack([x_all, z_all, tp_all]), delimiter=',', header='x,z,tp', comments='')
    print(f"Saved to {trajectory_filename_csv}.")

    # Save trajectories to .dat file in Tecplot format
    trajectory_filename_dat = os.path.join(result_folder, 'trajectories_x5300.dat')
    with open(trajectory_filename_dat, 'w') as f:
        f.write('VARIABLES = "X", "Z", "TP"\n')
        f.write(f"ZONE T='matrix_particle_trajectories_2d_tp0'\n")
        f.write(f'I={len(x_all)}, J=1, K=1, F=POINT\n')
        np.savetxt(f, np.column_stack([x_all, z_all, tp_all]), fmt='%f', delimiter='\t')
    print(f"Particle trajectories have also been saved to {trajectory_filename_dat}.")

    # Load stagnation points from CSV
    stagnation_points_path = os.path.join(result_folder, 'all_stagnation_points.csv')
    df_stagnation = pd.read_csv(stagnation_points_path)

    fig = go.Figure()

    for traj in all_trajectories:
        x_traj, z_traj, tp_traj = traj[:, 0], traj[:, 1], traj[:, 2]
        
        # Add particle trajectories
        fig.add_trace(go.Scatter(x=x_traj, y=z_traj, mode='lines+markers',
                                 marker=dict(size=6, color=tp_traj, colorscale=[[0, 'rgba(0, 255, 0, 1)'], [0.5, 'rgba(255, 255, 0, 0.5)'], [1, 'rgba(255, 0, 0, 1)']],
                                 colorbar=dict(title='Time (tp)'))))
        
        # Add starting points as black filled circles
        fig.add_trace(go.Scatter(x=[x_traj[0]], y=[z_traj[0]], mode='markers',
                                 marker=dict(size=12, color='black', symbol='circle', line=dict(width=2, color='white')),
                                 name='Starting Points'))

#     # Add stagnation points as black circles
#     x_stag, z_stag = df_stagnation['x'], df_stagnation['z']
#     fig.add_trace(go.Scatter(x=x_stag, y=z_stag, mode='markers',
#                              marker=dict(size=10, color='black', symbol='circle-open'),
#                              name='Stagnation Points'))
    
#     # Draw stagnation points closed loop
#     fig.add_trace(go.Scatter(x=np.append(x_stag, x_stag[0]), y=np.append(z_stag, z_stag[0]), mode='lines',
#                              line=dict(color='black', width=2, dash='dash'),
#                              name='Stagnation Closed Loop'))

    fig.update_layout(title="Particle Trajectories with Stagnation Points",
                      xaxis_title='X',
                      yaxis_title='Z',
                      margin=dict(r=10, l=10, b=10, t=40),
                      xaxis=dict(
                          range=[5200, 5400],
                          dtick=50,  
                          scaleanchor="y",  
                          scaleratio=1      
                      ),
                      yaxis=dict(range=[-400, 0]),
                      showlegend=False)


#     html_filename = os.path.join(result_folder, 'trajectories_x5300.html')
#     fig.write_html(html_filename)
#     print(f"Saved to {html_filename}.")
    fig.show()

if __name__ == "__main__":
    main()


In [ ]:
#Step 7-3: Generate local-scale trajectories across specified 1 point.without stagnant zone.x=5100
import numpy as np
import os
import pandas as pd
from pathlib import Path
import h5py
from numba import jit, prange
import plotly.graph_objects as go

# Parameter settings (ensure these are already defined)
params = {
    'p': 3, 'q': 3, 'HRx': 6, 'HLx': 3, 'D': -2000, 'a': 2/3, 'b': 1/3,
    'LX': 10000, 'LY': 10000, 'TauD': 0.4
}

# Extract parameters
p, q, HRx, HLx, D, a, b, TauD = params['p'], params['q'], params['HRx'], params['HLx'], params['D'], params['a'], params['b'], params['TauD']
LX, LY = params['LX'], params['LY']

# Define the mesh grid (only for x and z directions)
mesh_x, mesh_z = 101, 101
x_range = np.linspace(0, LX, mesh_x)
z_range = np.linspace(D, 0, mesh_z)

# Precompute cosine values
b1 = np.cos(np.pi * x_range / LX)
b2 = np.cos(p * np.pi * x_range / LX)

@jit(nopython=True, parallel=True)
def compute_J(mesh_x, mesh_z, x_range, z_range, HRx, HLx, LX, D, a, b, TauD, b1, b2, c1, c2, e1, e2):
    J = np.zeros((mesh_x, mesh_z))
    for i in prange(mesh_x):
        for k in prange(mesh_z):
            hA = HRx - HRx * b1[i] * np.cosh(np.pi * (z_range[k] - D) / LX) / c1
            hB = HLx - HLx * b2[i] * np.cosh(p * np.pi * (z_range[k] - D) / LX) / c2
            g1, g2 = 0.0, 0.0
            for n in range(1, 1000):  
                k1, an = (-1) ** (n - 1), (2 * n - 1) * np.pi / 2
                b0 = np.cos(an * (z_range[k] - D) / D)
                f1 = D * (k1 * an * b0 * (1 / an**2 + (an**2 * e1 + 2 * np.pi * TauD * e2) / (an**4 + 4 * np.pi**2 * TauD**2)))
                g1 += f1
                k2, d1 = (-1) ** n, an**2 + (p * np.pi * D / LX)**2
                f2 = D * (k2 * an * b0 * (1 / d1 + (d1 * e1 + 2 * np.pi * TauD * e2) / (d1**2 + 4 * np.pi**2 * TauD**2)))
                g2 += f2               
            hC = HLx * g1 / D + HLx * b2[i] * g2 / D
            J[i, k] = hA + a * hB + b * hC
    return J

def calculate_fields(tp):
    e1 = np.cos(2 * np.pi * tp)
    e2 = np.sin(2 * np.pi * tp)
    c1 = np.cosh(np.pi * D / LX)
    c2 = np.cosh(p * np.pi * D / LX)
    J = compute_J(mesh_x, mesh_z, x_range, z_range, HRx, HLx, LX, D, a, b, TauD, b1, b2, c1, c2, e1, e2)
    dx, dz = x_range[1] - x_range[0], z_range[1] - z_range[0]
    grad_x, grad_z = np.gradient(J, dx, dz)
    vx, vz = -100000*grad_x, -100000*grad_z
    UU = np.sqrt(vx**2 + vz**2)
    return J, vx, vz, UU

# Function to calculate the velocity at a given point (x, z) for a given tp
def get_velocity_at_point(x, z, tp):
    """Calculate the velocity components at a specific point (x, z) at time tp."""
    # Get the velocity field at the current time point
    H, vx, vz, UU = calculate_fields(tp)
    
    # Find the closest indices for (x, z) in the grid
    ix = np.abs(x_range - x).argmin()
    iz = np.abs(z_range - z).argmin()
    
    # Return the velocity components at the closest point
    return vx[ix, iz], vz[ix, iz]

def track_particle(x_start, z_start, tp_start, tp_end, step_size):
    x, z = x_start, z_start
    positions = [(x, z, tp_start)]
    for tp in np.arange(tp_start + step_size, tp_end, step_size):
        vx, vz = get_velocity_at_point(x, z, tp)
        x_new = x + vx * step_size
        z_new = z + vz * step_size
        if z_new >= 0:
            break
        positions.append((x_new, z_new, tp))
        x, z = x_new, z_new
    return np.array(positions)

def main():
    tp_start, tp_end, step_size = 0, 10.1, 0.1
    x_start_values = [5100]
    z_start_values = [-310]
    all_trajectories = []
    for x_start in x_start_values:
        for z_start in z_start_values:
            trajectory = track_particle(x_start, z_start, tp_start, tp_end, step_size)
            all_trajectories.append(trajectory)

    current_working_dir = os.getcwd()
    result_folder = os.path.join(current_working_dir, 'results')
    os.makedirs(result_folder, exist_ok=True)
    
    # Save trajectories to CSV
    x_all = np.concatenate([traj[:, 0] for traj in all_trajectories])
    z_all = np.concatenate([traj[:, 1] for traj in all_trajectories])
    tp_all = np.concatenate([traj[:, 2] for traj in all_trajectories])
    trajectory_filename_csv = os.path.join(result_folder, 'trajectories_x5100.csv')
    np.savetxt(trajectory_filename_csv, np.column_stack([x_all, z_all, tp_all]), delimiter=',', header='x,z,tp', comments='')
    print(f"Saved to {trajectory_filename_csv}.")

    # Save trajectories to .dat file in Tecplot format
    trajectory_filename_dat = os.path.join(result_folder, 'trajectories_x5100.dat')
    with open(trajectory_filename_dat, 'w') as f:
        f.write('VARIABLES = "X", "Z", "TP"\n')
        f.write(f"ZONE T='matrix_particle_trajectories_2d_tp0'\n")
        f.write(f'I={len(x_all)}, J=1, K=1, F=POINT\n')
        np.savetxt(f, np.column_stack([x_all, z_all, tp_all]), fmt='%f', delimiter='\t')
    print(f"Particle trajectories have also been saved to {trajectory_filename_dat}.")

    # Load stagnation points from CSV
    stagnation_points_path = os.path.join(result_folder, 'all_stagnation_points.csv')
    df_stagnation = pd.read_csv(stagnation_points_path)

    fig = go.Figure()

    for traj in all_trajectories:
        x_traj, z_traj, tp_traj = traj[:, 0], traj[:, 1], traj[:, 2]
        
        # Add particle trajectories
        fig.add_trace(go.Scatter(x=x_traj, y=z_traj, mode='lines+markers',
                                 marker=dict(size=5, color=tp_traj, colorscale=[[0, 'rgba(0, 255, 0, 1)'], [0.5, 'rgba(255, 255, 0, 0.5)'], [1, 'rgba(255, 0, 0, 1)']],
                                 colorbar=dict(title='Time (tp)'))))
        
        # Add starting points as black filled circles
        fig.add_trace(go.Scatter(x=[x_traj[0]], y=[z_traj[0]], mode='markers',
                                 marker=dict(size=8, color='black', symbol='circle', line=dict(width=2, color='white')),
                                 name='Starting Points'))

    # Add stagnation points as black circles
    x_stag, z_stag = df_stagnation['x'], df_stagnation['z']
    fig.add_trace(go.Scatter(x=x_stag, y=z_stag, mode='markers',
                             marker=dict(size=10, color='black', symbol='circle-open'),
                             name='Stagnation Points'))
    
    # Draw stagnation points closed loop
    fig.add_trace(go.Scatter(x=np.append(x_stag, x_stag[0]), y=np.append(z_stag, z_stag[0]), mode='lines',
                             line=dict(color='black', width=2, dash='dash'),
                             name='Stagnation Closed Loop'))

    fig.update_layout(title="Particle Trajectories with Stagnation Points",
                      xaxis_title='X',
                      yaxis_title='Z',
                      margin=dict(r=10, l=10, b=10, t=40),
                      xaxis=dict(
#                          range=[4500, 5500],
                          dtick=50,  
                          scaleanchor="y",  
                          scaleratio=1      
                      ),
                      yaxis=dict(range=[-350, -100]),
                      showlegend=False)

#     html_filename = os.path.join(result_folder, 'trajectories_x5100.html')
#     fig.write_html(html_filename)
#     print(f"Saved to {html_filename}.")
    fig.show()

if __name__ == "__main__":
    main()


In [ ]:
#Step 7-4: Generate local-scale trajectories across specified points.without stagnant zone.1 point at x=5000
import numpy as np
import os
import pandas as pd
from pathlib import Path
import h5py
from numba import jit, prange
import plotly.graph_objects as go

# Parameter settings (ensure these are already defined)
params = {
    'p': 3, 'q': 3, 'HRx': 6, 'HLx': 3, 'D': -2000, 'a': 2/3, 'b': 1/3,
    'LX': 10000, 'LY': 10000, 'TauD': 0.4
}

# Extract parameters
p, q, HRx, HLx, D, a, b, TauD = params['p'], params['q'], params['HRx'], params['HLx'], params['D'], params['a'], params['b'], params['TauD']
LX, LY = params['LX'], params['LY']

# Define the mesh grid (only for x and z directions)
mesh_x, mesh_z = 101, 101
x_range = np.linspace(0, LX, mesh_x)
z_range = np.linspace(D, 0, mesh_z)

# Precompute cosine values
b1 = np.cos(np.pi * x_range / LX)
b2 = np.cos(p * np.pi * x_range / LX)

@jit(nopython=True, parallel=True)
def compute_J(mesh_x, mesh_z, x_range, z_range, HRx, HLx, LX, D, a, b, TauD, b1, b2, c1, c2, e1, e2):
    J = np.zeros((mesh_x, mesh_z))
    for i in prange(mesh_x):
        for k in prange(mesh_z):
            hA = HRx - HRx * b1[i] * np.cosh(np.pi * (z_range[k] - D) / LX) / c1
            hB = HLx - HLx * b2[i] * np.cosh(p * np.pi * (z_range[k] - D) / LX) / c2
            g1, g2 = 0.0, 0.0
            for n in range(1, 1000):  
                k1, an = (-1) ** (n - 1), (2 * n - 1) * np.pi / 2
                b0 = np.cos(an * (z_range[k] - D) / D)
                f1 = D * (k1 * an * b0 * (1 / an**2 + (an**2 * e1 + 2 * np.pi * TauD * e2) / (an**4 + 4 * np.pi**2 * TauD**2)))
                g1 += f1
                k2, d1 = (-1) ** n, an**2 + (p * np.pi * D / LX)**2
                f2 = D * (k2 * an * b0 * (1 / d1 + (d1 * e1 + 2 * np.pi * TauD * e2) / (d1**2 + 4 * np.pi**2 * TauD**2)))
                g2 += f2               
            hC = HLx * g1 / D + HLx * b2[i] * g2 / D
            J[i, k] = hA + a * hB + b * hC
    return J

def calculate_fields(tp):
    e1 = np.cos(2 * np.pi * tp)
    e2 = np.sin(2 * np.pi * tp)
    c1 = np.cosh(np.pi * D / LX)
    c2 = np.cosh(p * np.pi * D / LX)
    J = compute_J(mesh_x, mesh_z, x_range, z_range, HRx, HLx, LX, D, a, b, TauD, b1, b2, c1, c2, e1, e2)
    dx, dz = x_range[1] - x_range[0], z_range[1] - z_range[0]
    grad_x, grad_z = np.gradient(J, dx, dz)
    vx, vz = -100000*grad_x, -100000*grad_z
    UU = np.sqrt(vx**2 + vz**2)
    return J, vx, vz, UU

# Function to calculate the velocity at a given point (x, z) for a given tp
def get_velocity_at_point(x, z, tp):
    """Calculate the velocity components at a specific point (x, z) at time tp."""
    # Get the velocity field at the current time point
    H, vx, vz, UU = calculate_fields(tp)
    
    # Find the closest indices for (x, z) in the grid
    ix = np.abs(x_range - x).argmin()
    iz = np.abs(z_range - z).argmin()
    
    # Return the velocity components at the closest point
    return vx[ix, iz], vz[ix, iz]

def track_particle(x_start, z_start, tp_start, tp_end, step_size):
    x, z = x_start, z_start
    positions = [(x, z, tp_start)]
    for tp in np.arange(tp_start + step_size, tp_end, step_size):
        vx, vz = get_velocity_at_point(x, z, tp)
        x_new = x + vx * step_size
        z_new = z + vz * step_size
        if z_new >= 0:
            break
        positions.append((x_new, z_new, tp))
        x, z = x_new, z_new
    return np.array(positions)

def main():
    tp_start, tp_end, step_size = 0, 10.1, 0.1
    x_start_values = [5000]
    z_start_values = [-310]
    all_trajectories = []
    for x_start in x_start_values:
        for z_start in z_start_values:
            trajectory = track_particle(x_start, z_start, tp_start, tp_end, step_size)
            all_trajectories.append(trajectory)

    current_working_dir = os.getcwd()
    result_folder = os.path.join(current_working_dir, 'results')
    os.makedirs(result_folder, exist_ok=True)
    
    # Save trajectories to CSV
    x_all = np.concatenate([traj[:, 0] for traj in all_trajectories])
    z_all = np.concatenate([traj[:, 1] for traj in all_trajectories])
    tp_all = np.concatenate([traj[:, 2] for traj in all_trajectories])
    trajectory_filename_csv = os.path.join(result_folder, 'trajectories_x5000.csv')
    np.savetxt(trajectory_filename_csv, np.column_stack([x_all, z_all, tp_all]), delimiter=',', header='x,z,tp', comments='')
    print(f"Saved to {trajectory_filename_csv}.")

    # Save trajectories to .dat file in Tecplot format
    trajectory_filename_dat = os.path.join(result_folder, 'trajectories_x5000.dat')
    with open(trajectory_filename_dat, 'w') as f:
        f.write('VARIABLES = "X", "Z", "TP"\n')
        f.write(f"ZONE T='matrix_particle_trajectories_2d_tp0'\n")
        f.write(f'I={len(x_all)}, J=1, K=1, F=POINT\n')
        np.savetxt(f, np.column_stack([x_all, z_all, tp_all]), fmt='%f', delimiter='\t')
    print(f"Particle trajectories have also been saved to {trajectory_filename_dat}.")

    # Load stagnation points from CSV
    stagnation_points_path = os.path.join(result_folder, 'all_stagnation_points.csv')
    df_stagnation = pd.read_csv(stagnation_points_path)

    fig = go.Figure()

    for traj in all_trajectories:
        x_traj, z_traj, tp_traj = traj[:, 0], traj[:, 1], traj[:, 2]
        
        # Add particle trajectories
        fig.add_trace(go.Scatter(x=x_traj, y=z_traj, mode='lines+markers',
                                 marker=dict(size=5, color=tp_traj, colorscale=[[0, 'rgba(0, 255, 0, 1)'], [0.5, 'rgba(255, 255, 0, 0.5)'], [1, 'rgba(255, 0, 0, 1)']],
                                 colorbar=dict(title='Time (tp)'))))
        
        # Add starting points as black filled circles
        fig.add_trace(go.Scatter(x=[x_traj[0]], y=[z_traj[0]], mode='markers',
                                 marker=dict(size=8, color='black', symbol='circle', line=dict(width=2, color='white')),
                                 name='Starting Points'))

#     # Add stagnation points as black circles
#     x_stag, z_stag = df_stagnation['x'], df_stagnation['z']
#     fig.add_trace(go.Scatter(x=x_stag, y=z_stag, mode='markers',
#                              marker=dict(size=10, color='black', symbol='circle-open'),
#                              name='Stagnation Points'))
    
#     # Draw stagnation points closed loop
#     fig.add_trace(go.Scatter(x=np.append(x_stag, x_stag[0]), y=np.append(z_stag, z_stag[0]), mode='lines',
#                              line=dict(color='black', width=2, dash='dash'),
#                              name='Stagnation Closed Loop'))

    fig.update_layout(title="Particle Trajectories with Stagnation Points",
                      xaxis_title='X',
                      yaxis_title='Z',
                      margin=dict(r=10, l=10, b=10, t=40),
                      xaxis=dict(
#                          range=[4500, 5500],
                          dtick=5,  
                          scaleanchor="y",  
                          scaleratio=1      
                      ),
                      yaxis=dict(range=[-320, -290]),
                      showlegend=False)

#     html_filename = os.path.join(result_folder, 'trajectories_x5000.html')
#     fig.write_html(html_filename)
#     print(f"Saved to {html_filename}.")
    fig.show()

if __name__ == "__main__":
    main()
